# ARC-AGI-3 — Hybrid Explorer Agent (self-contained)

General, training-free interactive agent (MIT-0): perception (object segmentation,
counter/distractor masking) -> state-transition graph exploration -> motion-model avatar
navigation -> 5-tier click salience. The arcagi3 package is embedded in this notebook
(written to /kaggle/working) so there is no external dependency; the agent is fail-safe
(random fallback) so it always acts.


In [ ]:
# Install the ARC-AGI-3 toolkit + engine offline from the competition wheels.
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv


In [ ]:
# Write the self-contained arcagi3 package to /kaggle/working/arcagi3 (no dataset dep).
import base64, os, pathlib
os.makedirs('/kaggle/working/arcagi3', exist_ok=True)
PKG = {'__init__.py': 'IiIiQVJDLUFHSS0zIGNvbXBldGl0aW9uIGFnZW50IChBUkMgUHJpemUgMjAyNikuIiIiCg==', 'perception.py': 'IiIiUGVyY2VwdGlvbjogdHVybiBhIHJhdyBBUkMtQUdJLTMgZnJhbWUgaW50byBhbiBvYmplY3QtY2VudHJpYyBzdGF0ZS4KCkEgZnJhbWUgZnJvbSB0aGUgZW5naW5lIGlzIGFuIGludDggYXJyYXkgb2Ygc2hhcGUgKE4sIDY0LCA2NCkgaG9sZGluZyBvbmUgb3IgbW9yZQpzdWItZnJhbWVzIChhbmltYXRpb24vdHJhbnNpdGlvbiBzdGVwcykgd2l0aCBjb2xvciB2YWx1ZXMgMC0xNS4gVGhlIGFnZW50IHJlYXNvbnMgb3Zlcgp0aGUgZmluYWwgc2V0dGxlZCBzdWItZnJhbWUgcGx1cyBhIHRlbXBvcmFsbHktZGVyaXZlZCBtYXNrIG9mICJ2b2xhdGlsZSIgY2VsbHMgKHN0YXR1cwpiYXJzIC8gY291bnRlcnMpIHRoYXQgbXVzdCBiZSBpZ25vcmVkIHdoZW4gZGVjaWRpbmcgd2hldGhlciB0d28gc3RhdGVzIGFyZSB0aGUgc2FtZS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBkZXF1ZQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCmZyb20gZnVuY3Rvb2xzIGltcG9ydCBscnVfY2FjaGUKCmltcG9ydCBudW1weSBhcyBucAoKR1JJRCA9IDY0CgoKZGVmIHRvX2dyaWQoZnJhbWUpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJSZXR1cm4gdGhlIGZpbmFsIHNldHRsZWQgNjR4NjQgc3ViLWZyYW1lIGFzIGFuIGludDggbmRhcnJheS4KCiAgICBBY2NlcHRzIGEgRnJhbWVEYXRhLCBhIGxpc3QsIG9yIGFuIG5kYXJyYXkgb2Ygc2hhcGUgKE4sNjQsNjQpIC8gKDY0LDY0KS4KICAgICIiIgogICAgaWYgaGFzYXR0cihmcmFtZSwgImZyYW1lIik6ICAjIGEgRnJhbWVEYXRhCiAgICAgICAgZnJhbWUgPSBmcmFtZS5mcmFtZQogICAgYXJyID0gbnAuYXNhcnJheShmcmFtZSwgZHR5cGU9bnAuaW50OCkKICAgIGlmIGFyci5uZGltID09IDM6CiAgICAgICAgYXJyID0gYXJyWy0xXQogICAgaWYgYXJyLm5kaW0gIT0gMjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYidW5leHBlY3RlZCBmcmFtZSBzaGFwZSB7YXJyLnNoYXBlfSIpCiAgICByZXR1cm4gYXJyCgoKZGVmIGdyaWRfc3RhY2soZnJhbWUpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJSZXR1cm4gYWxsIHN1Yi1mcmFtZXMgYXMgKE4sNjQsNjQpOyBhbmltYXRpb24gYWNyb3NzIGEgc2luZ2xlIHN0ZXAuIiIiCiAgICBpZiBoYXNhdHRyKGZyYW1lLCAiZnJhbWUiKToKICAgICAgICBmcmFtZSA9IGZyYW1lLmZyYW1lCiAgICBhcnIgPSBucC5hc2FycmF5KGZyYW1lLCBkdHlwZT1ucC5pbnQ4KQogICAgaWYgYXJyLm5kaW0gPT0gMjoKICAgICAgICBhcnIgPSBhcnJbTm9uZSwgOiwgOl0KICAgIHJldHVybiBhcnIKCgpkZWYgZGV0ZWN0X2JhY2tncm91bmQoZ3JpZDogbnAubmRhcnJheSkgLT4gaW50OgogICAgIiIiTW9zdCBmcmVxdWVudCBjb2xvciA9IHByZXN1bWVkIGJhY2tncm91bmQuIiIiCiAgICB2YWxzLCBjb3VudHMgPSBucC51bmlxdWUoZ3JpZCwgcmV0dXJuX2NvdW50cz1UcnVlKQogICAgcmV0dXJuIGludCh2YWxzW2ludChucC5hcmdtYXgoY291bnRzKSldKQoKCkBkYXRhY2xhc3MKY2xhc3MgT2JqOgogICAgIiIiQSBjb25uZWN0ZWQgcmVnaW9uIG9mIGEgc2luZ2xlIGNvbG9yICg0LWNvbm5lY3Rpdml0eSkuIiIiCgogICAgY29sb3I6IGludAogICAgY2VsbHM6IHR1cGxlW3R1cGxlW2ludCwgaW50XSwgLi4uXSAgIyAocm93LCBjb2wpIHBhaXJzCiAgICBiYm94OiB0dXBsZVtpbnQsIGludCwgaW50LCBpbnRdICAjIChyMCwgYzAsIHIxLCBjMSkgaW5jbHVzaXZlCiAgICBzaXplOiBpbnQKICAgIGNlbnRyb2lkOiB0dXBsZVtmbG9hdCwgZmxvYXRdCgogICAgQHByb3BlcnR5CiAgICBkZWYgdG9wX2xlZnQoc2VsZikgLT4gdHVwbGVbaW50LCBpbnRdOgogICAgICAgIHJldHVybiAoc2VsZi5iYm94WzBdLCBzZWxmLmJib3hbMV0pCgogICAgQHByb3BlcnR5CiAgICBkZWYgd2lkdGgoc2VsZikgLT4gaW50OgogICAgICAgIHJldHVybiBzZWxmLmJib3hbM10gLSBzZWxmLmJib3hbMV0gKyAxCgogICAgQHByb3BlcnR5CiAgICBkZWYgaGVpZ2h0KHNlbGYpIC0+IGludDoKICAgICAgICByZXR1cm4gc2VsZi5iYm94WzJdIC0gc2VsZi5iYm94WzBdICsgMQoKCmRlZiBjb25uZWN0ZWRfY29tcG9uZW50cygKICAgIGdyaWQ6IG5wLm5kYXJyYXksCiAgICBiYWNrZ3JvdW5kOiBpbnQgfCBOb25lID0gTm9uZSwKICAgIGluY2x1ZGVfYmFja2dyb3VuZDogYm9vbCA9IEZhbHNlLAopIC0+IGxpc3RbT2JqXToKICAgICIiIjQtY29ubmVjdGl2aXR5IGNvbm5lY3RlZCBjb21wb25lbnRzIG9mIGVxdWFsIGNvbG9yIChtZW1vaXplZCBwZXIgZ3JpZCkuCgogICAgQmFja2dyb3VuZCBjb2xvciBjb21wb25lbnRzIGFyZSBza2lwcGVkIHVubGVzcyBpbmNsdWRlX2JhY2tncm91bmQgaXMgVHJ1ZS4gUmVzdWx0cyBhcmUKICAgIGNhY2hlZCBvbiB0aGUgcmF3IGdyaWQgYnl0ZXM6IGEgc2luZ2xlIGRlY2lzaW9uIHN0ZXAgY2FsbHMgdGhpcyBzZXZlcmFsIHRpbWVzIG9uIHRoZQogICAgU0FNRSBncmlkIChzdGF0ZSBoYXNoaW5nLCBjbGljayB0YXJnZXRzLCBuYXYgdGFyZ2V0aW5nKSwgc28gbWVtb2l6aW5nIGlzIGEgcHVyZQogICAgc3BlZWR1cCAoaWRlbnRpY2FsIHJlc3VsdHMpIHRoYXQgYnV5cyBtb3JlIGFjdGlvbnMvc2VjIOKAlCBpLmUuIG1vcmUgbGV2ZWxzIGF0IGV2YWwuCiAgICAiIiIKICAgIGlmIGJhY2tncm91bmQgaXMgTm9uZToKICAgICAgICBiYWNrZ3JvdW5kID0gZGV0ZWN0X2JhY2tncm91bmQoZ3JpZCkKICAgIHJldHVybiBfY29ubmVjdGVkX2NvbXBvbmVudHNfY2FjaGVkKAogICAgICAgIG5wLmFzY29udGlndW91c2FycmF5KGdyaWQpLnRvYnl0ZXMoKSwgZ3JpZC5zaGFwZSwgaW50KGJhY2tncm91bmQpLCBpbmNsdWRlX2JhY2tncm91bmQKICAgICkKCgpAbHJ1X2NhY2hlKG1heHNpemU9MTYpCmRlZiBfY29ubmVjdGVkX2NvbXBvbmVudHNfY2FjaGVkKGdyaWRfYnl0ZXMsIHNoYXBlLCBiYWNrZ3JvdW5kLCBpbmNsdWRlX2JhY2tncm91bmQpIC0+IGxpc3RbT2JqXToKICAgIGdyaWQgPSBucC5mcm9tYnVmZmVyKGdyaWRfYnl0ZXMsIGR0eXBlPW5wLmludDgpLnJlc2hhcGUoc2hhcGUpCiAgICBoLCB3ID0gc2hhcGUKICAgIHNlZW4gPSBucC56ZXJvcygoaCwgdyksIGR0eXBlPWJvb2wpCiAgICBvYmpzOiBsaXN0W09ial0gPSBbXQogICAgZm9yIHIgaW4gcmFuZ2UoaCk6CiAgICAgICAgZm9yIGMgaW4gcmFuZ2Uodyk6CiAgICAgICAgICAgIGlmIHNlZW5bciwgY106CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBjb2xvciA9IGludChncmlkW3IsIGNdKQogICAgICAgICAgICBpZiBub3QgaW5jbHVkZV9iYWNrZ3JvdW5kIGFuZCBjb2xvciA9PSBiYWNrZ3JvdW5kOgogICAgICAgICAgICAgICAgc2VlbltyLCBjXSA9IFRydWUKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICMgQkZTIGZsb29kIGZpbGwKICAgICAgICAgICAgY2VsbHM6IGxpc3RbdHVwbGVbaW50LCBpbnRdXSA9IFtdCiAgICAgICAgICAgIHEgPSBkZXF1ZShbKHIsIGMpXSkKICAgICAgICAgICAgc2VlbltyLCBjXSA9IFRydWUKICAgICAgICAgICAgcjAgPSByMSA9IHIKICAgICAgICAgICAgYzAgPSBjMSA9IGMKICAgICAgICAgICAgc3IgPSBzYyA9IDAKICAgICAgICAgICAgd2hpbGUgcToKICAgICAgICAgICAgICAgIGNyLCBjYyA9IHEucG9wbGVmdCgpCiAgICAgICAgICAgICAgICBjZWxscy5hcHBlbmQoKGNyLCBjYykpCiAgICAgICAgICAgICAgICBzciArPSBjcgogICAgICAgICAgICAgICAgc2MgKz0gY2MKICAgICAgICAgICAgICAgIHIwLCByMSA9IG1pbihyMCwgY3IpLCBtYXgocjEsIGNyKQogICAgICAgICAgICAgICAgYzAsIGMxID0gbWluKGMwLCBjYyksIG1heChjMSwgY2MpCiAgICAgICAgICAgICAgICBmb3IgZHIsIGRjIGluICgoMSwgMCksICgtMSwgMCksICgwLCAxKSwgKDAsIC0xKSk6CiAgICAgICAgICAgICAgICAgICAgbnIsIG5jID0gY3IgKyBkciwgY2MgKyBkYwogICAgICAgICAgICAgICAgICAgIGlmIDAgPD0gbnIgPCBoIGFuZCAwIDw9IG5jIDwgdyBhbmQgbm90IHNlZW5bbnIsIG5jXSBhbmQgaW50KGdyaWRbbnIsIG5jXSkgPT0gY29sb3I6CiAgICAgICAgICAgICAgICAgICAgICAgIHNlZW5bbnIsIG5jXSA9IFRydWUKICAgICAgICAgICAgICAgICAgICAgICAgcS5hcHBlbmQoKG5yLCBuYykpCiAgICAgICAgICAgIG4gPSBsZW4oY2VsbHMpCiAgICAgICAgICAgIG9ianMuYXBwZW5kKAogICAgICAgICAgICAgICAgT2JqKAogICAgICAgICAgICAgICAgICAgIGNvbG9yPWNvbG9yLAogICAgICAgICAgICAgICAgICAgIGNlbGxzPXR1cGxlKGNlbGxzKSwKICAgICAgICAgICAgICAgICAgICBiYm94PShyMCwgYzAsIHIxLCBjMSksCiAgICAgICAgICAgICAgICAgICAgc2l6ZT1uLAogICAgICAgICAgICAgICAgICAgIGNlbnRyb2lkPShzciAvIG4sIHNjIC8gbiksCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICkKICAgIHJldHVybiBvYmpzCgoKZGVmIHN0YXRlX2hhc2goZ3JpZDogbnAubmRhcnJheSwgbWFzazogbnAubmRhcnJheSB8IE5vbmUgPSBOb25lKSAtPiBieXRlczoKICAgICIiIkV4YWN0IGhhc2ggb2YgdGhlIGdyaWQsIG9wdGlvbmFsbHkgemVyb2luZyBtYXNrZWQgKHZvbGF0aWxlKSBjZWxscyBmaXJzdC4iIiIKICAgIGlmIG1hc2sgaXMgbm90IE5vbmU6CiAgICAgICAgZyA9IGdyaWQuY29weSgpCiAgICAgICAgZ1ttYXNrXSA9IC0xCiAgICAgICAgcmV0dXJuIGcudG9ieXRlcygpCiAgICByZXR1cm4gbnAuYXNjb250aWd1b3VzYXJyYXkoZ3JpZCkudG9ieXRlcygpCgoKZGVmIG9iamVjdF9zdGF0ZV9rZXkoZ3JpZDogbnAubmRhcnJheSwgYmFja2dyb3VuZDogaW50IHwgTm9uZSA9IE5vbmUsCiAgICAgICAgICAgICAgICAgICAgIGlnbm9yZV9jb2xvcnM6IHNldFtpbnRdIHwgTm9uZSA9IE5vbmUpIC0+IGJ5dGVzOgogICAgIiIiQ29hcnNlLCByb2J1c3Qgc3RhdGUga2V5IGZyb20gT0JKRUNUIHN0cnVjdHVyZSAobm90IHJhdyBwaXhlbHMpLgoKICAgIEVhY2ggbm9uLWJhY2tncm91bmQsIG5vbi1pZ25vcmVkIGNvbm5lY3RlZCBjb21wb25lbnQgaXMgc3VtbWFyaXNlZCBhcwogICAgKGNvbG9yLCByMCwgYzAsIHIxLCBjMSwgc2l6ZSkuIFNvcnRpbmcgKyBzZXJpYWxpc2luZyB0aGVzZSBpcyBmYXIgbW9yZSBzdGFibGUgdGhhbiBhCiAgICBwaXhlbCBoYXNoOiBpdCBjb2xsYXBzZXMgd2l0aGluLW9iamVjdCBqaXR0ZXIgYW5kIGlycmVsZXZhbnQgc2luZ2xlLXBpeGVsIG5vaXNlIHRoYXQKICAgIHdvdWxkIG90aGVyd2lzZSBleHBsb2RlIHRoZSBzdGF0ZSBncmFwaCBvbiByZWFsIGdhbWVzLCB3aGlsZSBzdGlsbCBkaXN0aW5ndWlzaGluZwogICAgb2JqZWN0IG1vdmVzLCBhcHBlYXJhbmNlcy9kaXNhcHBlYXJhbmNlcywgYW5kIHNoYXBlIGNoYW5nZXMuIE1hdGNoZXMgdGhlIFNPVEEncwogICAgb2JqZWN0LXNlZ21lbnRhdGlvbiBhcHByb2FjaC4KICAgICIiIgogICAgaWYgYmFja2dyb3VuZCBpcyBOb25lOgogICAgICAgIGJhY2tncm91bmQgPSBkZXRlY3RfYmFja2dyb3VuZChncmlkKQogICAgaWdub3JlID0gaWdub3JlX2NvbG9ycyBvciBzZXQoKQogICAgcGFydHMgPSBbXQogICAgZm9yIG8gaW4gY29ubmVjdGVkX2NvbXBvbmVudHMoZ3JpZCwgYmFja2dyb3VuZD1iYWNrZ3JvdW5kKToKICAgICAgICBpZiBvLmNvbG9yIGluIGlnbm9yZToKICAgICAgICAgICAgY29udGludWUKICAgICAgICByMCwgYzAsIHIxLCBjMSA9IG8uYmJveAogICAgICAgIHBhcnRzLmFwcGVuZCgoby5jb2xvciwgcjAsIGMwLCByMSwgYzEsIG8uc2l6ZSkpCiAgICBwYXJ0cy5zb3J0KCkKICAgIHJldHVybiByZXByKHBhcnRzKS5lbmNvZGUoKQoKCmNsYXNzIFZvbGF0aWxpdHlUcmFja2VyOgogICAgIiIiVHJhY2tzIHdoaWNoIGNlbGxzIGNoYW5nZSBmcmVxdWVudGx5IGFjcm9zcyBzdGVwcyB0byBtYXNrIHN0YXR1cyBiYXJzL2NvdW50ZXJzLgoKICAgIENlbGxzIHRoYXQgY2hhbmdlIG9uIChhbG1vc3QpIGV2ZXJ5IHN0ZXAgcmVnYXJkbGVzcyBvZiBlZmZlY3QgYXJlIGxpa2VseSBzdGVwCiAgICBjb3VudGVycyBvciBhbmltYXRlZCBkZWNvcmF0aW9ucyBhbmQgc2hvdWxkIGJlIGV4Y2x1ZGVkIGZyb20gdGhlIHN0YXRlIGtleSBzbyB0aGUKICAgIHN0YXRlIGdyYXBoIGRvZXNuJ3QgZXhwbG9kZS4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCB0aHJlc2hvbGQ6IGZsb2F0ID0gMC45LCBtaW5fc3RlcHM6IGludCA9IDgpIC0+IE5vbmU6CiAgICAgICAgc2VsZi50aHJlc2hvbGQgPSB0aHJlc2hvbGQKICAgICAgICBzZWxmLm1pbl9zdGVwcyA9IG1pbl9zdGVwcwogICAgICAgIHNlbGYuY2hhbmdlczogbnAubmRhcnJheSB8IE5vbmUgPSBOb25lICAjIGxhemlseSBzaXplZCB0byB0aGUgYWN0dWFsIGZyYW1lCiAgICAgICAgc2VsZi5zaGFwZTogdHVwbGVbaW50LCBpbnRdID0gKEdSSUQsIEdSSUQpCiAgICAgICAgc2VsZi5zdGVwcyA9IDAKICAgICAgICBzZWxmLl9wcmV2OiBucC5uZGFycmF5IHwgTm9uZSA9IE5vbmUKCiAgICBkZWYgdXBkYXRlKHNlbGYsIGdyaWQ6IG5wLm5kYXJyYXkpIC0+IE5vbmU6CiAgICAgICAgIyBMYXppbHkgYWRvcHQgdGhlIHJlYWwgZnJhbWUgc2hhcGU7IHJlc2V0IGlmIGl0IGV2ZXIgY2hhbmdlcyAoZGVmZW5zaXZlKS4KICAgICAgICBpZiBzZWxmLmNoYW5nZXMgaXMgTm9uZSBvciBncmlkLnNoYXBlICE9IHNlbGYuc2hhcGU6CiAgICAgICAgICAgIHNlbGYuc2hhcGUgPSBncmlkLnNoYXBlCiAgICAgICAgICAgIHNlbGYuY2hhbmdlcyA9IG5wLnplcm9zKHNlbGYuc2hhcGUsIGR0eXBlPW5wLmludDMyKQogICAgICAgICAgICBzZWxmLnN0ZXBzID0gMAogICAgICAgICAgICBzZWxmLl9wcmV2ID0gTm9uZQogICAgICAgIGlmIHNlbGYuX3ByZXYgaXMgbm90IE5vbmUgYW5kIHNlbGYuX3ByZXYuc2hhcGUgPT0gZ3JpZC5zaGFwZToKICAgICAgICAgICAgc2VsZi5jaGFuZ2VzICs9IChncmlkICE9IHNlbGYuX3ByZXYpLmFzdHlwZShucC5pbnQzMikKICAgICAgICAgICAgc2VsZi5zdGVwcyArPSAxCiAgICAgICAgc2VsZi5fcHJldiA9IGdyaWQuY29weSgpCgogICAgZGVmIG1hc2soc2VsZikgLT4gbnAubmRhcnJheToKICAgICAgICAiIiJCb29sZWFuIG1hc2sgb2YgY2VsbHMgdG8gaWdub3JlIChUcnVlID0gdm9sYXRpbGUpLiIiIgogICAgICAgIGlmIHNlbGYuY2hhbmdlcyBpcyBOb25lIG9yIHNlbGYuc3RlcHMgPCBzZWxmLm1pbl9zdGVwczoKICAgICAgICAgICAgcmV0dXJuIG5wLnplcm9zKHNlbGYuc2hhcGUsIGR0eXBlPWJvb2wpCiAgICAgICAgcmV0dXJuIChzZWxmLmNoYW5nZXMgLyBtYXgoc2VsZi5zdGVwcywgMSkpID49IHNlbGYudGhyZXNob2xkCgoKZGVmIHNhbGllbnRfY2xpY2tfdGFyZ2V0cygKICAgIGdyaWQ6IG5wLm5kYXJyYXksIGJhY2tncm91bmQ6IGludCB8IE5vbmUgPSBOb25lLCBtYXhfdGFyZ2V0czogaW50ID0gNjQsCiAgICBjb2Fyc2VfZ3JpZF9zdGVwOiBpbnQgPSAwLAopIC0+IGxpc3RbdHVwbGVbaW50LCBpbnQsIGludF1dOgogICAgIiIiUHJvcG9zZSAoeCwgeSwgcHJpb3JpdHkpIGNsaWNrIHRhcmdldHMgZnJvbSBvYmplY3QgZ2VvbWV0cnkuCgogICAgT2JqZWN0LWNlbnRyaWMgaW5zdGVhZCBvZiBicnV0ZS1mb3JjaW5nIGFsbCA0MDk2IHBpeGVscy4gUHJpb3JpdHkgaXMgYSBzYWxpZW5jZQogICAgdGllciAobG93ZXIgPSB0cnkgZmlyc3QpOiBzbWFsbCBkaXN0aW5jdCBvYmplY3RzIGFuZCB0aGVpciBjb3JuZXJzIGFyZSBtb3N0IGxpa2VseQogICAgaW50ZXJhY3RpdmUuIFJldHVybnMgKHg9Y29sLCB5PXJvdywgcHJpb3JpdHkpLgoKICAgIElmIGNvYXJzZV9ncmlkX3N0ZXAgPiAwLCBhbHNvIGFkZCBhIGNvYXJzZSBsYXR0aWNlIG9mIGxvdy1wcmlvcml0eSBmYWxsYmFjayB0YXJnZXRzCiAgICAoZXZlcnkgYGNvYXJzZV9ncmlkX3N0ZXBgIHBpeGVscykgc28gbGFyZ2UgY2xpY2sgYWN0aW9uLXNwYWNlcyAoZS5nLiBmdDA5J3MgfjQwOTYKICAgIHBvc2l0aW9ucykgd2hlcmUgdGhlIGdvYWwgY2VsbCBpc24ndCBhbiBvYmplY3QgY2VudHJvaWQgYXJlIHN0aWxsIHJlYWNoYWJsZS4KICAgICIiIgogICAgaWYgYmFja2dyb3VuZCBpcyBOb25lOgogICAgICAgIGJhY2tncm91bmQgPSBkZXRlY3RfYmFja2dyb3VuZChncmlkKQogICAgaCwgdyA9IGdyaWQuc2hhcGUKICAgIG9ianMgPSBjb25uZWN0ZWRfY29tcG9uZW50cyhncmlkLCBiYWNrZ3JvdW5kPWJhY2tncm91bmQpCiAgICAjIGNvbG9yIHJhcml0eTogcmFyZXIgY29sb3JzIGFyZSBtb3JlIGxpa2VseSBpbnRlcmFjdGl2ZSAoYnV0dG9ucy9pdGVtcykKICAgIGNvbG9yX2NvdW50czogZGljdFtpbnQsIGludF0gPSB7fQogICAgZm9yIG8gaW4gb2JqczoKICAgICAgICBjb2xvcl9jb3VudHNbby5jb2xvcl0gPSBjb2xvcl9jb3VudHMuZ2V0KG8uY29sb3IsIDApICsgMQogICAgdGFyZ2V0czogbGlzdFt0dXBsZVtpbnQsIGludCwgaW50XV0gPSBbXQogICAgZm9yIG8gaW4gb2JqczoKICAgICAgICByLCBjID0gby5jZW50cm9pZAogICAgICAgIGNyLCBjYyA9IGludChyb3VuZChyKSksIGludChyb3VuZChjKSkKICAgICAgICByMCwgYzAsIHIxLCBjMSA9IG8uYmJveAogICAgICAgICMgU09UQS1zdHlsZSA1IHNhbGllbmNlIHRpZXJzIChsb3dlciA9IHRyeSBmaXJzdCk6IHNtYWxsICsgcmFyZS1jb2xvciBvYmplY3RzIGFyZQogICAgICAgICMgdGhlIG1vc3QgbGlrZWx5IGludGVyYWN0aXZlIGVsZW1lbnRzOyB3aWRlIGZsYXQgZWRnZS1odWdnaW5nIGJsb2JzIChzdGF0dXMgYmFycykKICAgICAgICAjIGdvIGxhc3QuCiAgICAgICAgaXNfc3RhdHVzX2JhciA9IChvLmhlaWdodCA8PSAyIG9yIG8ud2lkdGggPD0gMikgYW5kIChvLndpZHRoID49IHcgKiAwLjYgb3Igby5oZWlnaHQgPj0gaCAqIDAuNikKICAgICAgICByYXJlID0gY29sb3JfY291bnRzLmdldChvLmNvbG9yLCA5KSA8PSAyCiAgICAgICAgaWYgaXNfc3RhdHVzX2JhcjoKICAgICAgICAgICAgcHJpbyA9IDQKICAgICAgICBlbGlmIG8uc2l6ZSA8PSA0OgogICAgICAgICAgICBwcmlvID0gMCBpZiByYXJlIGVsc2UgMQogICAgICAgIGVsaWYgby5zaXplIDw9IDE2OgogICAgICAgICAgICBwcmlvID0gMSBpZiByYXJlIGVsc2UgMgogICAgICAgIGVsaWYgby5zaXplIDw9IDY0OgogICAgICAgICAgICBwcmlvID0gMiBpZiByYXJlIGVsc2UgMwogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHByaW8gPSAzCiAgICAgICAgdGFyZ2V0cy5hcHBlbmQoKGNjLCBjciwgcHJpbykpCiAgICAgICAgIyBjb3JuZXJzIG9mIGxhcmdlciBvYmplY3RzIChoYW5kbGVzL2VkZ2VzKSwgb25lIHRpZXIgbG93ZXIKICAgICAgICBpZiBvLnNpemUgPiA4IGFuZCBub3QgaXNfc3RhdHVzX2JhcjoKICAgICAgICAgICAgZm9yICh5eSwgeHgpIGluICgocjAsIGMwKSwgKHIwLCBjMSksIChyMSwgYzApLCAocjEsIGMxKSk6CiAgICAgICAgICAgICAgICB0YXJnZXRzLmFwcGVuZCgoeHgsIHl5LCBwcmlvICsgMSkpCiAgICAjIGNvYXJzZSBsYXR0aWNlIGZhbGxiYWNrIGZvciBsYXJnZSBjbGljayBzcGFjZXMgKGxvd2VzdCBwcmlvcml0eSkKICAgIGlmIGNvYXJzZV9ncmlkX3N0ZXAgYW5kIGNvYXJzZV9ncmlkX3N0ZXAgPiAwOgogICAgICAgIGgsIHcgPSBncmlkLnNoYXBlCiAgICAgICAgb2ZmID0gY29hcnNlX2dyaWRfc3RlcCAvLyAyCiAgICAgICAgZm9yIHl5IGluIHJhbmdlKG9mZiwgaCwgY29hcnNlX2dyaWRfc3RlcCk6CiAgICAgICAgICAgIGZvciB4eCBpbiByYW5nZShvZmYsIHcsIGNvYXJzZV9ncmlkX3N0ZXApOgogICAgICAgICAgICAgICAgdGFyZ2V0cy5hcHBlbmQoKHh4LCB5eSwgOSkpCiAgICAjIGRlZHVwIGtlZXBpbmcgYmVzdCAobG93ZXN0KSBwcmlvcml0eQogICAgYmVzdDogZGljdFt0dXBsZVtpbnQsIGludF0sIGludF0gPSB7fQogICAgZm9yIHgsIHksIHAgaW4gdGFyZ2V0czoKICAgICAgICBrID0gKHgsIHkpCiAgICAgICAgaWYgayBub3QgaW4gYmVzdCBvciBwIDwgYmVzdFtrXToKICAgICAgICAgICAgYmVzdFtrXSA9IHAKICAgIG91dCA9IFsoeCwgeSwgcCkgZm9yICh4LCB5KSwgcCBpbiBiZXN0Lml0ZW1zKCldCiAgICBvdXQuc29ydChrZXk9bGFtYmRhIHQ6IHRbMl0pCiAgICByZXR1cm4gb3V0WzptYXhfdGFyZ2V0c10K', 'world_model.py': 'IiIiV29ybGQgbW9kZWw6IGEgZGlyZWN0ZWQgZ3JhcGggb2Ygb2JzZXJ2ZWQgc3RhdGVzIGFuZCBhY3Rpb24gdHJhbnNpdGlvbnMuCgpOb2RlcyBhcmUgc3RhdGUga2V5cyAoaGFzaCBieXRlcyBvZiB0aGUgbWFza2VkIGdyaWQpLiBFZGdlcyByZWNvcmQsIGZvciBlYWNoIChzdGF0ZSwKYWN0aW9uKSBhY3R1YWxseSB0YWtlbiwgdGhlIHJlc3VsdGluZyBzdGF0ZSBhbmQgdGhlIHJld2FyZCAoY2hhbmdlIGluIGxldmVsc19jb21wbGV0ZWQpLgpBc3N1bWVzIChsb2NhbGx5KSBkZXRlcm1pbmlzdGljIGR5bmFtaWNzOiBzdGF0ZSArIGFjdGlvbiAtPiBzYW1lIG5leHQgc3RhdGUuIFRoZSBncmFwaApkcml2ZXMgZXhwbG9yYXRpb24gKGZpbmQgbmVhcmVzdCB1bmV4cGxvcmVkIGZyb250aWVyKSBhbmQgZXhwbG9pdGF0aW9uIChyZXBsYXkgYWN0aW9uCnNlcXVlbmNlcyB0aGF0IGxlYWQgdG8gcmV3YXJkKS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBkZXF1ZQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCmZyb20gdHlwaW5nIGltcG9ydCBIYXNoYWJsZSwgT3B0aW9uYWwKCiMgQW4gQWN0aW9uIGlzIGEgc21hbGwgaGFzaGFibGUgdG9rZW4gdGhlIGFnZW50IG1hcHMgdG8gYSBHYW1lQWN0aW9uOgojICAgKCJTIiwgYWN0aW9uX2lkKSAgICAgICAgICBzaW1wbGUgYWN0aW9uICgxLi41LCA3KQojICAgKCJDIiwgeCwgeSkgICAgICAgICAgICAgICBjb21wbGV4IGNsaWNrIChBQ1RJT042IGF0IHgseSkKQWN0aW9uID0gdHVwbGUKCgpAZGF0YWNsYXNzCmNsYXNzIE5vZGU6CiAgICBrZXk6IGJ5dGVzCiAgICBjYW5kaWRhdGVfYWN0aW9uczogdHVwbGVbQWN0aW9uLCAuLi5dID0gKCkgICMgZnVsbCBhY3Rpb24gc2V0IHByb3Bvc2VkIGF0IHRoaXMgc3RhdGUKICAgIGVkZ2VzOiBkaWN0W0FjdGlvbiwgdHVwbGVbYnl0ZXMsIGZsb2F0XV0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9ZGljdCkgICMgYWN0aW9uIC0+IChuZXh0X2tleSwgcmV3YXJkKQogICAgdGVybWluYWw6IGJvb2wgPSBGYWxzZSAgIyBHQU1FX09WRVIgcmVhY2hlZCBoZXJlIChvbmx5IFJFU0VUIGVzY2FwZXMpCiAgICB2aXNpdHM6IGludCA9IDAKCiAgICBkZWYgdW50cmllZChzZWxmKSAtPiBsaXN0W0FjdGlvbl06CiAgICAgICAgcmV0dXJuIFthIGZvciBhIGluIHNlbGYuY2FuZGlkYXRlX2FjdGlvbnMgaWYgYSBub3QgaW4gc2VsZi5lZGdlc10KCgpjbGFzcyBXb3JsZE1vZGVsOgogICAgZGVmIF9faW5pdF9fKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5ub2RlczogZGljdFtieXRlcywgTm9kZV0gPSB7fQoKICAgIGRlZiBvYnNlcnZlKHNlbGYsIGtleTogYnl0ZXMsIGNhbmRpZGF0ZXM6IHR1cGxlW0FjdGlvbiwgLi4uXSwgdGVybWluYWw6IGJvb2wgPSBGYWxzZSkgLT4gTm9kZToKICAgICAgICBub2RlID0gc2VsZi5ub2Rlcy5nZXQoa2V5KQogICAgICAgIGlmIG5vZGUgaXMgTm9uZToKICAgICAgICAgICAgbm9kZSA9IE5vZGUoa2V5PWtleSwgY2FuZGlkYXRlX2FjdGlvbnM9Y2FuZGlkYXRlcywgdGVybWluYWw9dGVybWluYWwpCiAgICAgICAgICAgIHNlbGYubm9kZXNba2V5XSA9IG5vZGUKICAgICAgICBlbHNlOgogICAgICAgICAgICAjIG1lcmdlIGFueSBuZXdseS1wcm9wb3NlZCBjYW5kaWRhdGVzIChrZWVwIG9yZGVyLCBkZWR1cCkKICAgICAgICAgICAgaWYgY2FuZGlkYXRlcyBhbmQgY2FuZGlkYXRlcyAhPSBub2RlLmNhbmRpZGF0ZV9hY3Rpb25zOgogICAgICAgICAgICAgICAgc2VlbiA9IHNldChub2RlLmNhbmRpZGF0ZV9hY3Rpb25zKQogICAgICAgICAgICAgICAgbWVyZ2VkID0gbGlzdChub2RlLmNhbmRpZGF0ZV9hY3Rpb25zKSArIFtjIGZvciBjIGluIGNhbmRpZGF0ZXMgaWYgYyBub3QgaW4gc2Vlbl0KICAgICAgICAgICAgICAgIG5vZGUuY2FuZGlkYXRlX2FjdGlvbnMgPSB0dXBsZShtZXJnZWQpCiAgICAgICAgICAgIG5vZGUudGVybWluYWwgPSBub2RlLnRlcm1pbmFsIG9yIHRlcm1pbmFsCiAgICAgICAgbm9kZS52aXNpdHMgKz0gMQogICAgICAgIHJldHVybiBub2RlCgogICAgZGVmIHJlY29yZChzZWxmLCBrZXk6IGJ5dGVzLCBhY3Rpb246IEFjdGlvbiwgbmV4dF9rZXk6IGJ5dGVzLCByZXdhcmQ6IGZsb2F0KSAtPiBOb25lOgogICAgICAgIG5vZGUgPSBzZWxmLm5vZGVzLmdldChrZXkpCiAgICAgICAgaWYgbm9kZSBpcyBOb25lOgogICAgICAgICAgICBub2RlID0gTm9kZShrZXk9a2V5KQogICAgICAgICAgICBzZWxmLm5vZGVzW2tleV0gPSBub2RlCiAgICAgICAgbm9kZS5lZGdlc1thY3Rpb25dID0gKG5leHRfa2V5LCByZXdhcmQpCgogICAgZGVmIGhhc191bnRyaWVkKHNlbGYsIGtleTogYnl0ZXMpIC0+IGJvb2w6CiAgICAgICAgbiA9IHNlbGYubm9kZXMuZ2V0KGtleSkKICAgICAgICByZXR1cm4gYm9vbChuIGFuZCBub3Qgbi50ZXJtaW5hbCBhbmQgbi51bnRyaWVkKCkpCgogICAgZGVmIHJld2FyZF9hY3Rpb24oc2VsZiwga2V5OiBieXRlcykgLT4gT3B0aW9uYWxbQWN0aW9uXToKICAgICAgICAiIiJJZiB0aGlzIHN0YXRlIGhhcyBhIGtub3duIGFjdGlvbiB0aGF0IHlpZWxkZWQgcG9zaXRpdmUgcmV3YXJkLCByZXR1cm4gaXQuIiIiCiAgICAgICAgbiA9IHNlbGYubm9kZXMuZ2V0KGtleSkKICAgICAgICBpZiBub3QgbjoKICAgICAgICAgICAgcmV0dXJuIE5vbmUKICAgICAgICBiZXN0LCBiZXN0X3IgPSBOb25lLCAwLjAKICAgICAgICBmb3IgYSwgKF9uaywgcikgaW4gbi5lZGdlcy5pdGVtcygpOgogICAgICAgICAgICBpZiByID4gYmVzdF9yOgogICAgICAgICAgICAgICAgYmVzdCwgYmVzdF9yID0gYSwgcgogICAgICAgIHJldHVybiBiZXN0CgogICAgZGVmIHBhdGhfdG9fZnJvbnRpZXIoc2VsZiwgc3RhcnQ6IGJ5dGVzLCBtYXhfZGVwdGg6IGludCA9IDEwMDAwMCkgLT4gT3B0aW9uYWxbbGlzdFtBY3Rpb25dXToKICAgICAgICAiIiJCRlMgb3ZlciBrbm93biBlZGdlcyB0byB0aGUgbmVhcmVzdCBub24tdGVybWluYWwgbm9kZSB3aXRoIHVudHJpZWQgYWN0aW9ucy4KCiAgICAgICAgUmV0dXJucyB0aGUgYWN0aW9uIHNlcXVlbmNlIGZyb20gYHN0YXJ0YCB0byB0aGF0IGZyb250aWVyIG5vZGUgKGVtcHR5IGxpc3QgaWYKICAgICAgICBgc3RhcnRgIGl0c2VsZiBpcyBhIGZyb250aWVyKSwgb3IgTm9uZSBpZiBub25lIHJlYWNoYWJsZS4KICAgICAgICAiIiIKICAgICAgICBpZiBzZWxmLmhhc191bnRyaWVkKHN0YXJ0KToKICAgICAgICAgICAgcmV0dXJuIFtdCiAgICAgICAgdmlzaXRlZCA9IHtzdGFydH0KICAgICAgICAjIHF1ZXVlIG9mIChrZXksIHBhdGgpCiAgICAgICAgcTogZGVxdWVbdHVwbGVbYnl0ZXMsIGxpc3RbQWN0aW9uXV1dID0gZGVxdWUoWyhzdGFydCwgW10pXSkKICAgICAgICB3aGlsZSBxOgogICAgICAgICAgICBrZXksIHBhdGggPSBxLnBvcGxlZnQoKQogICAgICAgICAgICBpZiBsZW4ocGF0aCkgPiBtYXhfZGVwdGg6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBub2RlID0gc2VsZi5ub2Rlcy5nZXQoa2V5KQogICAgICAgICAgICBpZiBub3Qgbm9kZToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGZvciBhY3Rpb24sIChuaywgX3IpIGluIG5vZGUuZWRnZXMuaXRlbXMoKToKICAgICAgICAgICAgICAgIGlmIG5rIGluIHZpc2l0ZWQ6CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIHZpc2l0ZWQuYWRkKG5rKQogICAgICAgICAgICAgICAgbnBhdGggPSBwYXRoICsgW2FjdGlvbl0KICAgICAgICAgICAgICAgIGlmIHNlbGYuaGFzX3VudHJpZWQobmspOgogICAgICAgICAgICAgICAgICAgIHJldHVybiBucGF0aAogICAgICAgICAgICAgICAgcS5hcHBlbmQoKG5rLCBucGF0aCkpCiAgICAgICAgcmV0dXJuIE5vbmUKCiAgICBkZWYgcGF0aF9iZXR3ZWVuKHNlbGYsIHN0YXJ0OiBieXRlcywgZ29hbDogYnl0ZXMpIC0+IE9wdGlvbmFsW2xpc3RbQWN0aW9uXV06CiAgICAgICAgIiIiU2hvcnRlc3Qga25vd24gYWN0aW9uIHBhdGggZnJvbSBzdGFydCB0byBnb2FsLCBvciBOb25lLiIiIgogICAgICAgIGlmIHN0YXJ0ID09IGdvYWw6CiAgICAgICAgICAgIHJldHVybiBbXQogICAgICAgIHZpc2l0ZWQgPSB7c3RhcnR9CiAgICAgICAgcTogZGVxdWVbdHVwbGVbYnl0ZXMsIGxpc3RbQWN0aW9uXV1dID0gZGVxdWUoWyhzdGFydCwgW10pXSkKICAgICAgICB3aGlsZSBxOgogICAgICAgICAgICBrZXksIHBhdGggPSBxLnBvcGxlZnQoKQogICAgICAgICAgICBub2RlID0gc2VsZi5ub2Rlcy5nZXQoa2V5KQogICAgICAgICAgICBpZiBub3Qgbm9kZToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGZvciBhY3Rpb24sIChuaywgX3IpIGluIG5vZGUuZWRnZXMuaXRlbXMoKToKICAgICAgICAgICAgICAgIGlmIG5rIGluIHZpc2l0ZWQ6CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIGlmIG5rID09IGdvYWw6CiAgICAgICAgICAgICAgICAgICAgcmV0dXJuIHBhdGggKyBbYWN0aW9uXQogICAgICAgICAgICAgICAgdmlzaXRlZC5hZGQobmspCiAgICAgICAgICAgICAgICBxLmFwcGVuZCgobmssIHBhdGggKyBbYWN0aW9uXSkpCiAgICAgICAgcmV0dXJuIE5vbmUKCiAgICBkZWYgX19sZW5fXyhzZWxmKSAtPiBpbnQ6CiAgICAgICAgcmV0dXJuIGxlbihzZWxmLm5vZGVzKQo=', 'movement.py': 'IiIiTW90aW9uIG1vZGVsOiBsZWFybiB0aGUgY29udHJvbGxhYmxlIG9iamVjdCAoYXZhdGFyKSBhbmQgaG93IGFjdGlvbnMgbW92ZSBpdC4KCk1vc3QgQVJDLUFHSS0zIGdhbWVzIChhbmQgaW50ZXJhY3RpdmUgZ2FtZXMgZ2VuZXJhbGx5KSBoYXZlIGFuIGF2YXRhciB0aGUgcGxheWVyIG1vdmVzCndpdGggc2ltcGxlIGFjdGlvbnMuIElmIHdlIGNhbiBpZGVudGlmeSBpdCBhbmQgbGVhcm4gZWFjaCBhY3Rpb24ncyBkaXNwbGFjZW1lbnQgdmVjdG9yLAp3ZSBjYW4gbmF2aWdhdGUgaW4gY29vcmRpbmF0ZSBzcGFjZSAoY2hlYXAsIGdvYWwtZGlyZWN0ZWQpIGluc3RlYWQgb2YgYmxpbmQgZXhwbG9yYXRpb24Kb3ZlciBoYXNoZWQgc3RhdGVzLiBUaGlzIGlzIGZ1bGx5IGdlbmVyYWwg4oCUIG5vIHBlci1nYW1lIGtub3dsZWRnZS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCgppbXBvcnQgbnVtcHkgYXMgbnAKCmZyb20gLiBpbXBvcnQgcGVyY2VwdGlvbiBhcyBQCgoKZGVmIGNvbG9yZWRfY2VsbHMoZ3JpZDogbnAubmRhcnJheSwgYmFja2dyb3VuZDogaW50KSAtPiBkaWN0W2ludCwgbnAubmRhcnJheV06CiAgICAiIiJNYXAgY29sb3IgLT4gYm9vbGVhbiBtYXNrIG9mIGl0cyBjZWxscyAoZXhjbHVkaW5nIGJhY2tncm91bmQpLiIiIgogICAgb3V0ID0ge30KICAgIGZvciBjIGluIG5wLnVuaXF1ZShncmlkKToKICAgICAgICBpZiBpbnQoYykgPT0gYmFja2dyb3VuZDoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBvdXRbaW50KGMpXSA9IGdyaWQgPT0gYwogICAgcmV0dXJuIG91dAoKCmRlZiBpbmZlcl90cmFuc2xhdGlvbihiZWZvcmU6IG5wLm5kYXJyYXksIGFmdGVyOiBucC5uZGFycmF5LCBiYWNrZ3JvdW5kOiBpbnQpOgogICAgIiIiSWYgZXhhY3RseSBvbmUgY29sb3IncyByZWdpb24gdHJhbnNsYXRlZCBieSBhIGNvbnN0YW50IHZlY3RvciwgcmV0dXJuIChjb2xvciwgZHIsIGRjKS4KCiAgICBSZXR1cm5zIE5vbmUgaWYgdGhlIGNoYW5nZSBpc24ndCBhIGNsZWFuIHNpbmdsZS1vYmplY3QgdHJhbnNsYXRpb24uCiAgICAiIiIKICAgIGlmIGJlZm9yZS5zaGFwZSAhPSBhZnRlci5zaGFwZToKICAgICAgICByZXR1cm4gTm9uZQogICAgY2hhbmdlZF9jb2xvcnMgPSBbXQogICAgZm9yIGMgaW4gc2V0KG5wLnVuaXF1ZShiZWZvcmUpKS51bmlvbihucC51bmlxdWUoYWZ0ZXIpKToKICAgICAgICBjID0gaW50KGMpCiAgICAgICAgaWYgYyA9PSBiYWNrZ3JvdW5kOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGIgPSBiZWZvcmUgPT0gYwogICAgICAgIGEgPSBhZnRlciA9PSBjCiAgICAgICAgaWYgbm90IG5wLmFycmF5X2VxdWFsKGIsIGEpOgogICAgICAgICAgICBjaGFuZ2VkX2NvbG9ycy5hcHBlbmQoYykKICAgICMgVGhlIGF2YXRhciBpcyBhIGNvbG9yIHdob3NlIG1hc2sgbW92ZWQuIFN0YXRpYyBkZWNvcmF0aW9ucyBkb24ndCBjaGFuZ2UuCiAgICBiZXN0ID0gTm9uZQogICAgZm9yIGMgaW4gY2hhbmdlZF9jb2xvcnM6CiAgICAgICAgYiA9IG5wLmFyZ3doZXJlKGJlZm9yZSA9PSBjKQogICAgICAgIGEgPSBucC5hcmd3aGVyZShhZnRlciA9PSBjKQogICAgICAgIGlmIGxlbihiKSA9PSAwIG9yIGxlbihhKSA9PSAwIG9yIGxlbihiKSAhPSBsZW4oYSk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgIyBjYW5kaWRhdGUgdHJhbnNsYXRpb24gPSBjZW50cm9pZCBzaGlmdAogICAgICAgIGRiID0gYi5tZWFuKGF4aXM9MCkKICAgICAgICBkYSA9IGEubWVhbihheGlzPTApCiAgICAgICAgZHIsIGRjID0gZGEgLSBkYiwgTm9uZQogICAgICAgIHNoaWZ0ID0gKGRhIC0gZGIpCiAgICAgICAgIyB2ZXJpZnkgaXQncyBhIHJpZ2lkIHRyYW5zbGF0aW9uOiBzaGlmdGluZyBiZWZvcmUtY2VsbHMgYnkgcm91bmQoc2hpZnQpID09IGFmdGVyLWNlbGxzCiAgICAgICAgc3IsIHNjID0gaW50KHJvdW5kKHNoaWZ0WzBdKSksIGludChyb3VuZChzaGlmdFsxXSkpCiAgICAgICAgc2hpZnRlZCA9IGIgKyBucC5hcnJheShbc3IsIHNjXSkKICAgICAgICBpZiBzZXQobWFwKHR1cGxlLCBzaGlmdGVkLnRvbGlzdCgpKSkgPT0gc2V0KG1hcCh0dXBsZSwgYS50b2xpc3QoKSkpOgogICAgICAgICAgICBpZiAoc3IsIHNjKSAhPSAoMCwgMCk6CiAgICAgICAgICAgICAgICAjIHByZWZlciB0aGUgc21hbGxlc3QgbW92aW5nIG9iamVjdCAobGlrZWx5IHRoZSBhdmF0YXIpCiAgICAgICAgICAgICAgICBpZiBiZXN0IGlzIE5vbmUgb3IgbGVuKGIpIDwgYmVzdFszXToKICAgICAgICAgICAgICAgICAgICBiZXN0ID0gKGMsIHNyLCBzYywgbGVuKGIpKQogICAgaWYgYmVzdCBpcyBOb25lOgogICAgICAgIHJldHVybiBOb25lCiAgICByZXR1cm4gKGJlc3RbMF0sIGJlc3RbMV0sIGJlc3RbMl0pCgoKZGVmIGluZmVyX2FsbF90cmFuc2xhdGlvbnMoYmVmb3JlOiBucC5uZGFycmF5LCBhZnRlcjogbnAubmRhcnJheSwgYmFja2dyb3VuZDogaW50KSAtPiBkaWN0OgogICAgIiIiUmV0dXJuIHtjb2xvcjogKGRyLCBkYyl9IGZvciBldmVyeSBub24tYmFja2dyb3VuZCBjb2xvciB0aGF0IHJpZ2lkbHkgdHJhbnNsYXRlZC4KCiAgICBVbmxpa2UgaW5mZXJfdHJhbnNsYXRpb24gKHNpbmdsZSBiZXN0IG1vdmVyKSwgdGhpcyByZXBvcnRzIGFsbCBtb3ZlcnMgc28gdGhlIGNhbGxlcgogICAgY2FuIGRpc3Rpbmd1aXNoIHRoZSBhdmF0YXIgKG1vdGlvbiB2YXJpZXMgd2l0aCB0aGUgYWN0aW9uKSBmcm9tIGluZGVwZW5kZW50CiAgICBhbmltYXRpb25zL2NvdW50ZXJzIChtb3Rpb24gaXMgY29uc3RhbnQgcmVnYXJkbGVzcyBvZiB0aGUgYWN0aW9uKS4KICAgICIiIgogICAgb3V0OiBkaWN0W2ludCwgdHVwbGVbaW50LCBpbnRdXSA9IHt9CiAgICBpZiBiZWZvcmUuc2hhcGUgIT0gYWZ0ZXIuc2hhcGU6CiAgICAgICAgcmV0dXJuIG91dAogICAgZm9yIGMgaW4gc2V0KG5wLnVuaXF1ZShiZWZvcmUpKS51bmlvbihucC51bmlxdWUoYWZ0ZXIpKToKICAgICAgICBjID0gaW50KGMpCiAgICAgICAgaWYgYyA9PSBiYWNrZ3JvdW5kOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGIgPSBucC5hcmd3aGVyZShiZWZvcmUgPT0gYykKICAgICAgICBhID0gbnAuYXJnd2hlcmUoYWZ0ZXIgPT0gYykKICAgICAgICBpZiBsZW4oYikgPT0gMCBvciBsZW4oYSkgPT0gMCBvciBsZW4oYikgIT0gbGVuKGEpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHNoaWZ0ID0gYS5tZWFuKGF4aXM9MCkgLSBiLm1lYW4oYXhpcz0wKQogICAgICAgIHNyLCBzYyA9IGludChyb3VuZChzaGlmdFswXSkpLCBpbnQocm91bmQoc2hpZnRbMV0pKQogICAgICAgIGlmIChzciwgc2MpID09ICgwLCAwKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBzaGlmdGVkID0gYiArIG5wLmFycmF5KFtzciwgc2NdKQogICAgICAgIGlmIHNldChtYXAodHVwbGUsIHNoaWZ0ZWQudG9saXN0KCkpKSA9PSBzZXQobWFwKHR1cGxlLCBhLnRvbGlzdCgpKSk6CiAgICAgICAgICAgIG91dFtjXSA9IChzciwgc2MpCiAgICByZXR1cm4gb3V0CgoKQGRhdGFjbGFzcwpjbGFzcyBNb3Rpb25Nb2RlbDoKICAgIGF2YXRhcl9jb2xvcjogaW50IHwgTm9uZSA9IE5vbmUgICMgcHJpbWFyeSBjb2xvciAoZm9yIGRlbHRhIGxvb2t1cCkKICAgIGRlbHRhczogZGljdFtpbnQsIHR1cGxlW2ludCwgaW50XV0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9ZGljdCkgICMgYWN0aW9uX2lkIC0+IChkcixkYykKICAgIGF2YXRhcl9jb2xvcnM6IGZyb3plbnNldCA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1mcm96ZW5zZXQpICAjIGFsbCBjb2xvcnMgbW92aW5nIGFzIHRoZSBhdmF0YXIKCiAgICBAcHJvcGVydHkKICAgIGRlZiBvayhzZWxmKSAtPiBib29sOgogICAgICAgIHJldHVybiBzZWxmLmF2YXRhcl9jb2xvciBpcyBub3QgTm9uZSBhbmQgbGVuKHNlbGYuZGVsdGFzKSA+IDAKCiAgICBkZWYgX21hc2soc2VsZiwgZ3JpZDogbnAubmRhcnJheSkgLT4gbnAubmRhcnJheToKICAgICAgICBjb2xzID0gc2VsZi5hdmF0YXJfY29sb3JzIG9yICh7c2VsZi5hdmF0YXJfY29sb3J9IGlmIHNlbGYuYXZhdGFyX2NvbG9yIGlzIG5vdCBOb25lIGVsc2Ugc2V0KCkpCiAgICAgICAgcmV0dXJuIG5wLmlzaW4oZ3JpZCwgbGlzdChjb2xzKSkKCiAgICBkZWYgYXZhdGFyX2NlbnRyb2lkKHNlbGYsIGdyaWQ6IG5wLm5kYXJyYXkpOgogICAgICAgICIiIkNlbnRyb2lkIChyb3csY29sKSBvdmVyIEFMTCBhdmF0YXIgY29sb3JzIChtdWx0aS1jb2xvciBhdmF0YXJzIG1vdmUgdG9nZXRoZXIpLiIiIgogICAgICAgIGNlbGxzID0gbnAuYXJnd2hlcmUoc2VsZi5fbWFzayhncmlkKSkKICAgICAgICBpZiBsZW4oY2VsbHMpID09IDA6CiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgcmV0dXJuIHR1cGxlKGNlbGxzLm1lYW4oYXhpcz0wKSkKCiAgICBkZWYgYXZhdGFyX2NlbGxzKHNlbGYsIGdyaWQ6IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgcmV0dXJuIG5wLmFyZ3doZXJlKHNlbGYuX21hc2soZ3JpZCkpCg==', 'agent.py': 'IiIiQWdlbnRzIGZvciBBUkMtQUdJLTMuCgotIEdyYXBoU3RyYXRlZ3k6IGdyYXBoLWJhc2VkIGV4cGxvcmF0aW9uL2V4cGxvaXRhdGlvbiBvdmVyIGEgV29ybGRNb2RlbCAoZnJvbnRpZXIgc2VhcmNoCiAgKyBzaG9ydGVzdC1wYXRoIHJlcGxheSArIHJld2FyZCBleHBsb2l0YXRpb24pLiBSZXVzYWJsZSBkZWNpc2lvbiBwb2xpY3kuCi0gRXhwbG9yZXJBZ2VudDogcHVyZSBncmFwaCBleHBsb3JlciAoYmFzZWxpbmUpLgotIEh5YnJpZEFnZW50OiBsZWFybnMgYSBtb3Rpb24gbW9kZWwgKGNvbnRyb2xsYWJsZSBhdmF0YXIgKyBwZXItYWN0aW9uIGRpc3BsYWNlbWVudCkgYW5kCiAgbmF2aWdhdGVzIGluIGNvb3JkaW5hdGUgc3BhY2UgdG8gY2FuZGlkYXRlIGdvYWwgb2JqZWN0czsgZmFsbHMgYmFjayB0byBHcmFwaFN0cmF0ZWd5CiAgd2hlbiBubyBhdmF0YXIgaXMgZm91bmQgb3IgbW90aW9uIHByb2dyZXNzIHN0YWxscy4KCkFsbCB0cmFpbmluZy1mcmVlIGFuZCBnYW1lLWFnbm9zdGljLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBsb2dnaW5nCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwoKaW1wb3J0IG51bXB5IGFzIG5wCmZyb20gYXJjZW5naW5lIGltcG9ydCBHYW1lQWN0aW9uLCBHYW1lU3RhdGUKCmZyb20gLiBpbXBvcnQgbW92ZW1lbnQgYXMgTVYKZnJvbSAuIGltcG9ydCBwZXJjZXB0aW9uIGFzIFAKZnJvbSAud29ybGRfbW9kZWwgaW1wb3J0IEFjdGlvbiwgV29ybGRNb2RlbAoKbG9nZ2VyID0gbG9nZ2luZy5nZXRMb2dnZXIoImFyY2FnaTMuYWdlbnQiKQoKU0lNUExFX0lEUyA9IFsxLCAyLCAzLCA0LCA1LCA3XSAgIyBSRVNFVCgwKS9BQ1RJT042KGNsaWNrKSBoYW5kbGVkIHNlcGFyYXRlbHkKCgpAZGF0YWNsYXNzCmNsYXNzIFBsYXlSZXN1bHQ6CiAgICBnYW1lX2lkOiBzdHIKICAgIGxldmVsc19jb21wbGV0ZWQ6IGludAogICAgd2luX2xldmVsczogaW50CiAgICBhY3Rpb25zOiBpbnQKICAgIHdvbjogYm9vbAogICAgc3RhdGVzX3NlZW46IGludAogICAgcmVhc29uOiBzdHIgPSAiIgoKCmRlZiB0b19nYW1lX2FjdGlvbihhOiBBY3Rpb24pIC0+IHR1cGxlW0dhbWVBY3Rpb24sIGRpY3RdOgogICAgaWYgYVswXSA9PSAiUyI6CiAgICAgICAgcmV0dXJuIEdhbWVBY3Rpb24uZnJvbV9pZChhWzFdKSwge30KICAgIGlmIGFbMF0gPT0gIkMiOgogICAgICAgIHJldHVybiBHYW1lQWN0aW9uLkFDVElPTjYsIHsieCI6IGFbMV0sICJ5IjogYVsyXX0KICAgIHJhaXNlIFZhbHVlRXJyb3IoYSkKCgpkZWYgY2FuZGlkYXRlc19mb3IoZ3JpZDogbnAubmRhcnJheSwgYXZhaWxhYmxlOiBsaXN0W2ludF0sIHVzZV9jbGlja3M6IGJvb2wsCiAgICAgICAgICAgICAgICAgICBtYXhfY2xpY2tfdGFyZ2V0czogaW50LCB1c2VfdW5kbzogYm9vbCkgLT4gdHVwbGVbQWN0aW9uLCAuLi5dOgogICAgY2FuZHM6IGxpc3RbQWN0aW9uXSA9IFtdCiAgICBmb3IgYWlkIGluIFNJTVBMRV9JRFM6CiAgICAgICAgaWYgYWlkIGluIGF2YWlsYWJsZSBhbmQgKGFpZCAhPSA3IG9yIHVzZV91bmRvKToKICAgICAgICAgICAgY2FuZHMuYXBwZW5kKCgiUyIsIGFpZCkpCiAgICBpZiB1c2VfY2xpY2tzIGFuZCA2IGluIGF2YWlsYWJsZToKICAgICAgICBmb3IgeCwgeSwgX3ByaW8gaW4gUC5zYWxpZW50X2NsaWNrX3RhcmdldHMoCiAgICAgICAgICAgIGdyaWQsIG1heF90YXJnZXRzPW1heF9jbGlja190YXJnZXRzLCBjb2Fyc2VfZ3JpZF9zdGVwPTgKICAgICAgICApOgogICAgICAgICAgICBjYW5kcy5hcHBlbmQoKCJDIiwgaW50KHgpLCBpbnQoeSkpKQogICAgcmV0dXJuIHR1cGxlKGNhbmRzKQoKCiMgLS0tIGRlY2lzaW9ucyB0aGUgc3RyYXRlZ3kgY2FuIHJldHVybiB0byB0aGUgZHJpdmluZyBsb29wIC0tLQpBQ1QgPSAiYWN0IgpSRVNFVCA9ICJyZXNldCIKU1RPUCA9ICJzdG9wIgoKCmNsYXNzIEdyYXBoU3RyYXRlZ3k6CiAgICAiIiJTdGF0ZWZ1bCBncmFwaCBleHBsb3JhdGlvbiBwb2xpY3kgb3ZlciBhIHNoYXJlZCBXb3JsZE1vZGVsLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCByb290X2tleTogYnl0ZXMsIG1heF9zdHVja19yZXNldHM6IGludCA9IDUwKSAtPiBOb25lOgogICAgICAgIHNlbGYud20gPSBXb3JsZE1vZGVsKCkKICAgICAgICBzZWxmLnJvb3Rfa2V5ID0gcm9vdF9rZXkKICAgICAgICBzZWxmLnBsYW46IGxpc3RbQWN0aW9uXSA9IFtdCiAgICAgICAgc2VsZi5fZXhwZWN0OiBieXRlcyB8IE5vbmUgPSBOb25lCiAgICAgICAgc2VsZi5zdHVja19yZXNldHMgPSAwCiAgICAgICAgc2VsZi5tYXhfc3R1Y2tfcmVzZXRzID0gbWF4X3N0dWNrX3Jlc2V0cwoKICAgIGRlZiBkZWNpZGUoc2VsZiwgY3VyX2tleTogYnl0ZXMpIC0+IHR1cGxlW3N0ciwgQWN0aW9uIHwgTm9uZV06CiAgICAgICAgbm9kZSA9IHNlbGYud20ubm9kZXMuZ2V0KGN1cl9rZXkpCiAgICAgICAgaWYgbm9kZSBpcyBOb25lOgogICAgICAgICAgICByZXR1cm4gKFNUT1AsIE5vbmUpCgogICAgICAgICMgMSkgZXhwbG9pdCBhIGtub3duIHJld2FyZC1wcm9kdWNpbmcgYWN0aW9uCiAgICAgICAgcl9hY3QgPSBzZWxmLndtLnJld2FyZF9hY3Rpb24oY3VyX2tleSkKICAgICAgICBpZiByX2FjdCBpcyBub3QgTm9uZToKICAgICAgICAgICAgcmV0dXJuIChBQ1QsIHJfYWN0KQoKICAgICAgICAjIDIpIGNvbnRpbnVlIGFuIGFjdGl2ZSBwbGFuIChyZXBsYXkpLCBhYm9ydGluZyBvbiBkaXZlcmdlbmNlCiAgICAgICAgaWYgc2VsZi5wbGFuOgogICAgICAgICAgICBpZiBzZWxmLl9leHBlY3QgaXMgbm90IE5vbmUgYW5kIGN1cl9rZXkgIT0gc2VsZi5fZXhwZWN0OgogICAgICAgICAgICAgICAgc2VsZi5wbGFuID0gW10KICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHJldHVybiAoQUNULCBzZWxmLnBsYW4ucG9wKDApKQoKICAgICAgICAjIDMpIHVudHJpZWQgY2FuZGlkYXRlIGhlcmUKICAgICAgICBpZiBub2RlLnVudHJpZWQoKToKICAgICAgICAgICAgcmV0dXJuIChBQ1QsIG5vZGUudW50cmllZCgpWzBdKQoKICAgICAgICAjIDQpIG5hdmlnYXRlIHRvIG5lYXJlc3QgZnJvbnRpZXIKICAgICAgICBwYXRoID0gc2VsZi53bS5wYXRoX3RvX2Zyb250aWVyKGN1cl9rZXkpCiAgICAgICAgaWYgcGF0aCBpcyBOb25lOgogICAgICAgICAgICBpZiBjdXJfa2V5ICE9IHNlbGYucm9vdF9rZXkgYW5kIHNlbGYuc3R1Y2tfcmVzZXRzIDwgc2VsZi5tYXhfc3R1Y2tfcmVzZXRzOgogICAgICAgICAgICAgICAgc2VsZi5zdHVja19yZXNldHMgKz0gMQogICAgICAgICAgICAgICAgcmV0dXJuIChSRVNFVCwgTm9uZSkKICAgICAgICAgICAgcGF0aCA9IHNlbGYud20ucGF0aF90b19mcm9udGllcihzZWxmLnJvb3Rfa2V5KQogICAgICAgICAgICBpZiBwYXRoIGlzIE5vbmU6CiAgICAgICAgICAgICAgICByZXR1cm4gKFNUT1AsIE5vbmUpCiAgICAgICAgaWYgcGF0aDoKICAgICAgICAgICAgc2VsZi5wbGFuID0gcGF0aAogICAgICAgICAgICByZXR1cm4gKEFDVCwgc2VsZi5wbGFuLnBvcCgwKSkKICAgICAgICByZXR1cm4gKFNUT1AsIE5vbmUpCgogICAgZGVmIHVwZGF0ZShzZWxmLCBjdXJfa2V5OiBieXRlcywgYWN0aW9uOiBBY3Rpb24sIG5leHRfa2V5OiBieXRlcywgcmV3YXJkOiBmbG9hdCwKICAgICAgICAgICAgICAgY2FuZGlkYXRlczogdHVwbGVbQWN0aW9uLCAuLi5dLCB0ZXJtaW5hbDogYm9vbCkgLT4gTm9uZToKICAgICAgICBzZWxmLndtLnJlY29yZChjdXJfa2V5LCBhY3Rpb24sIG5leHRfa2V5LCByZXdhcmQpCiAgICAgICAgc2VsZi53bS5vYnNlcnZlKG5leHRfa2V5LCBjYW5kaWRhdGVzLCB0ZXJtaW5hbD10ZXJtaW5hbCkKICAgICAgICBzZWxmLl9leHBlY3QgPSBuZXh0X2tleSBpZiBzZWxmLnBsYW4gZWxzZSBOb25lCgoKY2xhc3MgRXhwbG9yZXJBZ2VudDoKICAgICIiIlB1cmUgZ3JhcGgtYmFzZWQgZXhwbG9yZXIgKGJhc2VsaW5lKS4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgbWF4X2FjdGlvbnM6IGludCA9IDQwMDAsIHVzZV9jbGlja3M6IGJvb2wgPSBUcnVlLAogICAgICAgICAgICAgICAgIG1heF9jbGlja190YXJnZXRzOiBpbnQgPSA5NiwgdXNlX3VuZG86IGJvb2wgPSBGYWxzZSwgc2VlZDogaW50ID0gMCkgLT4gTm9uZToKICAgICAgICBzZWxmLm1heF9hY3Rpb25zID0gbWF4X2FjdGlvbnMKICAgICAgICBzZWxmLnVzZV9jbGlja3MgPSB1c2VfY2xpY2tzCiAgICAgICAgc2VsZi5tYXhfY2xpY2tfdGFyZ2V0cyA9IG1heF9jbGlja190YXJnZXRzCiAgICAgICAgc2VsZi51c2VfdW5kbyA9IHVzZV91bmRvCgogICAgZGVmIF9jYW5kcyhzZWxmLCBncmlkLCBhdmFpbGFibGUpOgogICAgICAgIHJldHVybiBjYW5kaWRhdGVzX2ZvcihncmlkLCBhdmFpbGFibGUsIHNlbGYudXNlX2NsaWNrcywgc2VsZi5tYXhfY2xpY2tfdGFyZ2V0cywgc2VsZi51c2VfdW5kbykKCiAgICBkZWYgcGxheShzZWxmLCBlbnYsIGdhbWVfaWQ6IHN0ciA9ICI/IikgLT4gUGxheVJlc3VsdDoKICAgICAgICB2dCA9IFAuVm9sYXRpbGl0eVRyYWNrZXIoKQogICAgICAgIG9icyA9IGVudi5yZXNldCgpCiAgICAgICAgZ3JpZCA9IFAudG9fZ3JpZChvYnMuZnJhbWUpCiAgICAgICAgdnQudXBkYXRlKGdyaWQpCiAgICAgICAgcm9vdF9rZXkgPSBQLnN0YXRlX2hhc2goZ3JpZCwgdnQubWFzaygpKQogICAgICAgIGdzID0gR3JhcGhTdHJhdGVneShyb290X2tleSkKICAgICAgICBncy53bS5vYnNlcnZlKHJvb3Rfa2V5LCBzZWxmLl9jYW5kcyhncmlkLCBvYnMuYXZhaWxhYmxlX2FjdGlvbnMpKQogICAgICAgIGN1cl9rZXkgPSByb290X2tleQogICAgICAgIGFjdGlvbnMgPSAwCiAgICAgICAgcHJldl9sZXZlbHMgPSBpbnQob2JzLmxldmVsc19jb21wbGV0ZWQgb3IgMCkKICAgICAgICB3aW5fbGV2ZWxzID0gaW50KG9icy53aW5fbGV2ZWxzIG9yIDApCiAgICAgICAgcmVhc29uID0gImJ1ZGdldCIKCiAgICAgICAgd2hpbGUgYWN0aW9ucyA8IHNlbGYubWF4X2FjdGlvbnM6CiAgICAgICAgICAgIGlmIG9icy5zdGF0ZSA9PSBHYW1lU3RhdGUuV0lOOgogICAgICAgICAgICAgICAgcmVhc29uID0gIndpbiIKICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgIGlmIG9icy5zdGF0ZSA9PSBHYW1lU3RhdGUuR0FNRV9PVkVSOgogICAgICAgICAgICAgICAgbiA9IGdzLndtLm5vZGVzLmdldChjdXJfa2V5KQogICAgICAgICAgICAgICAgaWYgbjoKICAgICAgICAgICAgICAgICAgICBuLnRlcm1pbmFsID0gVHJ1ZQogICAgICAgICAgICAgICAgb2JzID0gZW52LnJlc2V0KCk7IGFjdGlvbnMgKz0gMQogICAgICAgICAgICAgICAgZ3JpZCA9IFAudG9fZ3JpZChvYnMuZnJhbWUpOyB2dC51cGRhdGUoZ3JpZCk7IGN1cl9rZXkgPSByb290X2tleTsgZ3MucGxhbiA9IFtdCiAgICAgICAgICAgICAgICBjb250aW51ZQoKICAgICAgICAgICAga2luZCwgYWN0aW9uID0gZ3MuZGVjaWRlKGN1cl9rZXkpCiAgICAgICAgICAgIGlmIGtpbmQgPT0gU1RPUDoKICAgICAgICAgICAgICAgIHJlYXNvbiA9ICJleGhhdXN0ZWQiOyBicmVhawogICAgICAgICAgICBpZiBraW5kID09IFJFU0VUOgogICAgICAgICAgICAgICAgb2JzID0gZW52LnJlc2V0KCk7IGFjdGlvbnMgKz0gMQogICAgICAgICAgICAgICAgZ3JpZCA9IFAudG9fZ3JpZChvYnMuZnJhbWUpOyB2dC51cGRhdGUoZ3JpZCk7IGN1cl9rZXkgPSByb290X2tleTsgZ3MucGxhbiA9IFtdCiAgICAgICAgICAgICAgICBjb250aW51ZQoKICAgICAgICAgICAgb2JzLCBncmlkLCBjdXJfa2V5LCBwcmV2X2xldmVscyA9IHNlbGYuX3N0ZXAoZW52LCBhY3Rpb24sIGdzLCB2dCwgY3VyX2tleSwgcHJldl9sZXZlbHMpCiAgICAgICAgICAgIGFjdGlvbnMgKz0gMQoKICAgICAgICByZXR1cm4gUGxheVJlc3VsdChnYW1lX2lkLCBwcmV2X2xldmVscywgd2luX2xldmVscywgYWN0aW9ucywKICAgICAgICAgICAgICAgICAgICAgICAgICBvYnMuc3RhdGUgPT0gR2FtZVN0YXRlLldJTiwgbGVuKGdzLndtKSwgcmVhc29uKQoKICAgIGRlZiBfc3RlcChzZWxmLCBlbnYsIGFjdGlvbiwgZ3MsIHZ0LCBjdXJfa2V5LCBwcmV2X2xldmVscyk6CiAgICAgICAgZ2EsIGRhdGEgPSB0b19nYW1lX2FjdGlvbihhY3Rpb24pCiAgICAgICAgb2JzID0gZW52LnN0ZXAoZ2EsIGRhdGE9ZGF0YSkgaWYgZGF0YSBlbHNlIGVudi5zdGVwKGdhKQogICAgICAgIG5ncmlkID0gUC50b19ncmlkKG9icy5mcmFtZSk7IHZ0LnVwZGF0ZShuZ3JpZCkKICAgICAgICBua2V5ID0gUC5zdGF0ZV9oYXNoKG5ncmlkLCB2dC5tYXNrKCkpCiAgICAgICAgbmxldmVscyA9IGludChvYnMubGV2ZWxzX2NvbXBsZXRlZCBvciAwKQogICAgICAgIHRlcm1pbmFsID0gb2JzLnN0YXRlID09IEdhbWVTdGF0ZS5HQU1FX09WRVIKICAgICAgICBncy51cGRhdGUoY3VyX2tleSwgYWN0aW9uLCBua2V5LCBmbG9hdChubGV2ZWxzIC0gcHJldl9sZXZlbHMpLAogICAgICAgICAgICAgICAgICBzZWxmLl9jYW5kcyhuZ3JpZCwgb2JzLmF2YWlsYWJsZV9hY3Rpb25zKSwgdGVybWluYWwpCiAgICAgICAgcmV0dXJuIG9icywgbmdyaWQsIG5rZXksIG5sZXZlbHMKCgpjbGFzcyBIeWJyaWRBZ2VudDoKICAgICIiIk1vdGlvbi1maXJzdCBhZ2VudDogbGVhcm4gdGhlIGF2YXRhciArIHBlci1hY3Rpb24gZGlzcGxhY2VtZW50LCBuYXZpZ2F0ZSB0byBnb2FsCiAgICBvYmplY3RzIGluIGNvb3JkaW5hdGUgc3BhY2U7IGZhbGwgYmFjayB0byBncmFwaCBleHBsb3JhdGlvbiB3aGVuIHN0YWxsZWQuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIG1heF9hY3Rpb25zOiBpbnQgPSA0MDAwLCB1c2VfY2xpY2tzOiBib29sID0gVHJ1ZSwKICAgICAgICAgICAgICAgICBtYXhfY2xpY2tfdGFyZ2V0czogaW50ID0gOTYsIG5hdl9zdGVwX2NhcDogaW50ID0gMjAwLCBzZWVkOiBpbnQgPSAwKSAtPiBOb25lOgogICAgICAgIHNlbGYubWF4X2FjdGlvbnMgPSBtYXhfYWN0aW9ucwogICAgICAgIHNlbGYudXNlX2NsaWNrcyA9IHVzZV9jbGlja3MKICAgICAgICBzZWxmLm1heF9jbGlja190YXJnZXRzID0gbWF4X2NsaWNrX3RhcmdldHMKICAgICAgICBzZWxmLm5hdl9zdGVwX2NhcCA9IG5hdl9zdGVwX2NhcAogICAgICAgIHNlbGYucm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpCgogICAgZGVmIF9jYW5kcyhzZWxmLCBncmlkLCBhdmFpbGFibGUpOgogICAgICAgIHJldHVybiBjYW5kaWRhdGVzX2ZvcihncmlkLCBhdmFpbGFibGUsIHNlbGYudXNlX2NsaWNrcywgc2VsZi5tYXhfY2xpY2tfdGFyZ2V0cywgRmFsc2UpCgogICAgZGVmIHBsYXkoc2VsZiwgZW52LCBnYW1lX2lkOiBzdHIgPSAiPyIpIC0+IFBsYXlSZXN1bHQ6CiAgICAgICAgdnQgPSBQLlZvbGF0aWxpdHlUcmFja2VyKCkKICAgICAgICBvYnMgPSBlbnYucmVzZXQoKQogICAgICAgIGdyaWQgPSBQLnRvX2dyaWQob2JzLmZyYW1lKTsgdnQudXBkYXRlKGdyaWQpCiAgICAgICAgcm9vdF9rZXkgPSBQLnN0YXRlX2hhc2goZ3JpZCwgdnQubWFzaygpKQogICAgICAgIGdzID0gR3JhcGhTdHJhdGVneShyb290X2tleSkKICAgICAgICBncy53bS5vYnNlcnZlKHJvb3Rfa2V5LCBzZWxmLl9jYW5kcyhncmlkLCBvYnMuYXZhaWxhYmxlX2FjdGlvbnMpKQogICAgICAgIGN1cl9rZXkgPSByb290X2tleQogICAgICAgIGFjdGlvbnMgPSAwCiAgICAgICAgcHJldl9sZXZlbHMgPSBpbnQob2JzLmxldmVsc19jb21wbGV0ZWQgb3IgMCkKICAgICAgICB3aW5fbGV2ZWxzID0gaW50KG9icy53aW5fbGV2ZWxzIG9yIDApCiAgICAgICAgcmVhc29uID0gImJ1ZGdldCIKCiAgICAgICAgYmcgPSBQLmRldGVjdF9iYWNrZ3JvdW5kKGdyaWQpCiAgICAgICAgbW06IE1WLk1vdGlvbk1vZGVsIHwgTm9uZSA9IE5vbmUKICAgICAgICBsZXZlbF9vZl9tb2RlbCA9IC0xCiAgICAgICAgdHJpZWRfdGFyZ2V0czogc2V0W3R1cGxlW2ludCwgaW50XV0gPSBzZXQoKQogICAgICAgIG1vdGlvbl9kZWFkID0gRmFsc2UgICMgYXZhdGFyIHN0cmF0ZWd5IGdhdmUgdXAgZm9yIHRoaXMgbGV2ZWwKCiAgICAgICAgZGVmIHJlY29yZChhY3Rpb24sIG9ic19uZXcpOgogICAgICAgICAgICBub25sb2NhbCBjdXJfa2V5LCBwcmV2X2xldmVscywgZ3JpZCwgYWN0aW9ucwogICAgICAgICAgICBuZ3JpZCA9IFAudG9fZ3JpZChvYnNfbmV3LmZyYW1lKTsgdnQudXBkYXRlKG5ncmlkKQogICAgICAgICAgICBua2V5ID0gUC5zdGF0ZV9oYXNoKG5ncmlkLCB2dC5tYXNrKCkpCiAgICAgICAgICAgIG5sZXZlbHMgPSBpbnQob2JzX25ldy5sZXZlbHNfY29tcGxldGVkIG9yIDApCiAgICAgICAgICAgIHRlcm1pbmFsID0gb2JzX25ldy5zdGF0ZSA9PSBHYW1lU3RhdGUuR0FNRV9PVkVSCiAgICAgICAgICAgIGdzLnVwZGF0ZShjdXJfa2V5LCBhY3Rpb24sIG5rZXksIGZsb2F0KG5sZXZlbHMgLSBwcmV2X2xldmVscyksCiAgICAgICAgICAgICAgICAgICAgICBzZWxmLl9jYW5kcyhuZ3JpZCwgb2JzX25ldy5hdmFpbGFibGVfYWN0aW9ucyksIHRlcm1pbmFsKQogICAgICAgICAgICBjdXJfa2V5LCBwcmV2X2xldmVscywgZ3JpZCA9IG5rZXksIG5sZXZlbHMsIG5ncmlkCiAgICAgICAgICAgIGFjdGlvbnMgKz0gMQoKICAgICAgICB3aGlsZSBhY3Rpb25zIDwgc2VsZi5tYXhfYWN0aW9uczoKICAgICAgICAgICAgaWYgb2JzLnN0YXRlID09IEdhbWVTdGF0ZS5XSU46CiAgICAgICAgICAgICAgICByZWFzb24gPSAid2luIjsgYnJlYWsKICAgICAgICAgICAgaWYgb2JzLnN0YXRlID09IEdhbWVTdGF0ZS5HQU1FX09WRVI6CiAgICAgICAgICAgICAgICBuID0gZ3Mud20ubm9kZXMuZ2V0KGN1cl9rZXkpCiAgICAgICAgICAgICAgICBpZiBuOgogICAgICAgICAgICAgICAgICAgIG4udGVybWluYWwgPSBUcnVlCiAgICAgICAgICAgICAgICBvYnMgPSBlbnYucmVzZXQoKTsgYWN0aW9ucyArPSAxCiAgICAgICAgICAgICAgICBncmlkID0gUC50b19ncmlkKG9icy5mcmFtZSk7IHZ0LnVwZGF0ZShncmlkKTsgY3VyX2tleSA9IHJvb3Rfa2V5OyBncy5wbGFuID0gW10KICAgICAgICAgICAgICAgIG1tID0gTm9uZTsgbW90aW9uX2RlYWQgPSBGYWxzZTsgdHJpZWRfdGFyZ2V0cy5jbGVhcigpCiAgICAgICAgICAgICAgICBjb250aW51ZQoKICAgICAgICAgICAgIyBOZXcgbGV2ZWwgLT4gcmVsZWFybiBtb3Rpb24KICAgICAgICAgICAgaWYgcHJldl9sZXZlbHMgIT0gbGV2ZWxfb2ZfbW9kZWw6CiAgICAgICAgICAgICAgICBtbSA9IE5vbmU7IG1vdGlvbl9kZWFkID0gRmFsc2U7IHRyaWVkX3RhcmdldHMuY2xlYXIoKQogICAgICAgICAgICAgICAgbGV2ZWxfb2ZfbW9kZWwgPSBwcmV2X2xldmVscwoKICAgICAgICAgICAgc2ltcGxlX2F2YWlsID0gW2EgZm9yIGEgaW4gKDEsIDIsIDMsIDQsIDUpIGlmIGEgaW4gb2JzLmF2YWlsYWJsZV9hY3Rpb25zXQoKICAgICAgICAgICAgIyAtLS0tIGxlYXJuIG1vdGlvbiBtb2RlbCBieSBwcm9iaW5nIHNpbXBsZSBhY3Rpb25zIC0tLS0KICAgICAgICAgICAgaWYgbW0gaXMgTm9uZSBhbmQgc2ltcGxlX2F2YWlsIGFuZCBub3QgbW90aW9uX2RlYWQ6CiAgICAgICAgICAgICAgICBtbSA9IHNlbGYuX2xlYXJuX21vdGlvbihlbnYsIG9icywgZ3JpZCwgYmcsIHNpbXBsZV9hdmFpbCwgcmVjb3JkX2ZuPXJlY29yZCkKICAgICAgICAgICAgICAgIG9icyA9IHNlbGYuX2xhc3Rfb2JzCiAgICAgICAgICAgICAgICBpZiBtbSBpcyBOb25lIG9yIG5vdCBtbS5kZWx0YXM6CiAgICAgICAgICAgICAgICAgICAgbW90aW9uX2RlYWQgPSBUcnVlCiAgICAgICAgICAgICAgICBjb250aW51ZQoKICAgICAgICAgICAgIyAtLS0tIG5hdmlnYXRlIGF2YXRhciB0byBhIGNhbmRpZGF0ZSBnb2FsIG9iamVjdCAtLS0tCiAgICAgICAgICAgIGlmIG1tIGlzIG5vdCBOb25lIGFuZCBtbS5vayBhbmQgbm90IG1vdGlvbl9kZWFkOgogICAgICAgICAgICAgICAgdGFyZ2V0ID0gc2VsZi5fbmV4dF90YXJnZXQoZ3JpZCwgYmcsIG1tLCB0cmllZF90YXJnZXRzKQogICAgICAgICAgICAgICAgaWYgdGFyZ2V0IGlzIE5vbmU6CiAgICAgICAgICAgICAgICAgICAgbW90aW9uX2RlYWQgPSBUcnVlCiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIHRyaWVkX3RhcmdldHMuYWRkKHRhcmdldCkKICAgICAgICAgICAgICAgIG9icyA9IHNlbGYuX25hdmlnYXRlKGVudiwgbW0sIHRhcmdldCwgcmVjb3JkX2ZuPXJlY29yZCwgc3RhcnRfbGV2ZWxzPXByZXZfbGV2ZWxzKQogICAgICAgICAgICAgICAgaWYgb2JzLnN0YXRlID09IEdhbWVTdGF0ZS5XSU46CiAgICAgICAgICAgICAgICAgICAgcmVhc29uID0gIndpbiI7IGJyZWFrCiAgICAgICAgICAgICAgICBjb250aW51ZQoKICAgICAgICAgICAgIyAtLS0tIGdyYXBoIGZhbGxiYWNrIC0tLS0KICAgICAgICAgICAga2luZCwgYWN0aW9uID0gZ3MuZGVjaWRlKGN1cl9rZXkpCiAgICAgICAgICAgIGlmIGtpbmQgPT0gU1RPUDoKICAgICAgICAgICAgICAgICMgbGFzdCByZXNvcnQ6IGlmIG1vdGlvbiBleGlzdGVkLCByZXNldCBhbmQgbGV0IG1vdGlvbiByZXRyeSBmcmVzaAogICAgICAgICAgICAgICAgcmVhc29uID0gImV4aGF1c3RlZCI7IGJyZWFrCiAgICAgICAgICAgIGlmIGtpbmQgPT0gUkVTRVQ6CiAgICAgICAgICAgICAgICBvYnMgPSBlbnYucmVzZXQoKTsgYWN0aW9ucyArPSAxCiAgICAgICAgICAgICAgICBncmlkID0gUC50b19ncmlkKG9icy5mcmFtZSk7IHZ0LnVwZGF0ZShncmlkKTsgY3VyX2tleSA9IHJvb3Rfa2V5OyBncy5wbGFuID0gW10KICAgICAgICAgICAgICAgIG1tID0gTm9uZTsgbW90aW9uX2RlYWQgPSBGYWxzZTsgdHJpZWRfdGFyZ2V0cy5jbGVhcigpOyBsZXZlbF9vZl9tb2RlbCA9IHByZXZfbGV2ZWxzCiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBnYSwgZGF0YSA9IHRvX2dhbWVfYWN0aW9uKGFjdGlvbikKICAgICAgICAgICAgb2JzID0gZW52LnN0ZXAoZ2EsIGRhdGE9ZGF0YSkgaWYgZGF0YSBlbHNlIGVudi5zdGVwKGdhKQogICAgICAgICAgICByZWNvcmQoYWN0aW9uLCBvYnMpCgogICAgICAgIHJldHVybiBQbGF5UmVzdWx0KGdhbWVfaWQsIHByZXZfbGV2ZWxzLCB3aW5fbGV2ZWxzLCBhY3Rpb25zLAogICAgICAgICAgICAgICAgICAgICAgICAgIG9icy5zdGF0ZSA9PSBHYW1lU3RhdGUuV0lOLCBsZW4oZ3Mud20pLCByZWFzb24pCgogICAgIyAtLS0tLSBtb3Rpb24gbGVhcm5pbmcgLS0tLS0KICAgIGRlZiBfbGVhcm5fbW90aW9uKHNlbGYsIGVudiwgb2JzLCBncmlkLCBiZywgc2ltcGxlX2F2YWlsLCByZWNvcmRfZm4pIC0+IE1WLk1vdGlvbk1vZGVsIHwgTm9uZToKICAgICAgICAiIiJUcnkgZWFjaCBzaW1wbGUgYWN0aW9uIG9uY2U7IGRldGVjdCB0aGUgYXZhdGFyIChjb25zaXN0ZW50bHktdHJhbnNsYXRpbmcgY29sb3IpLiIiIgogICAgICAgIHZvdGVzOiBkaWN0W2ludCwgZGljdFtpbnQsIHR1cGxlW2ludCwgaW50XV1dID0ge30gICMgY29sb3IgLT4ge2FjdGlvbjogKGRyLGRjKX0KICAgICAgICBjdXJfZ3JpZCA9IGdyaWQKICAgICAgICBsYXN0X29icyA9IG9icwogICAgICAgIGZvciBhaWQgaW4gc2ltcGxlX2F2YWlsOgogICAgICAgICAgICBiZWZvcmUgPSBjdXJfZ3JpZAogICAgICAgICAgICBhY3Rpb24gPSAoIlMiLCBhaWQpCiAgICAgICAgICAgIGdhLCBfID0gdG9fZ2FtZV9hY3Rpb24oYWN0aW9uKQogICAgICAgICAgICBvID0gZW52LnN0ZXAoZ2EpCiAgICAgICAgICAgIHJlY29yZF9mbihhY3Rpb24sIG8pCiAgICAgICAgICAgIGxhc3Rfb2JzID0gbwogICAgICAgICAgICBhZnRlciA9IFAudG9fZ3JpZChvLmZyYW1lKQogICAgICAgICAgICByZXMgPSBNVi5pbmZlcl90cmFuc2xhdGlvbihiZWZvcmUsIGFmdGVyLCBiZykKICAgICAgICAgICAgaWYgcmVzIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgY29sb3IsIGRyLCBkYyA9IHJlcwogICAgICAgICAgICAgICAgdm90ZXMuc2V0ZGVmYXVsdChjb2xvciwge30pW2FpZF0gPSAoZHIsIGRjKQogICAgICAgICAgICBjdXJfZ3JpZCA9IGFmdGVyCiAgICAgICAgICAgIGlmIG8uc3RhdGUgaW4gKEdhbWVTdGF0ZS5XSU4sIEdhbWVTdGF0ZS5HQU1FX09WRVIpOgogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICBzZWxmLl9sYXN0X29icyA9IGxhc3Rfb2JzCiAgICAgICAgaWYgbm90IHZvdGVzOgogICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgICMgYXZhdGFyID0gY29sb3IgdGhhdCBtb3ZlZCBmb3IgdGhlIG1vc3QgYWN0aW9ucwogICAgICAgIGF2YXRhcl9jb2xvciA9IG1heCh2b3Rlcywga2V5PWxhbWJkYSBjOiBsZW4odm90ZXNbY10pKQogICAgICAgIHJldHVybiBNVi5Nb3Rpb25Nb2RlbChhdmF0YXJfY29sb3I9YXZhdGFyX2NvbG9yLCBkZWx0YXM9dm90ZXNbYXZhdGFyX2NvbG9yXSkKCiAgICBkZWYgX25leHRfdGFyZ2V0KHNlbGYsIGdyaWQsIGJnLCBtbTogTVYuTW90aW9uTW9kZWwsIHRyaWVkKSAtPiB0dXBsZVtpbnQsIGludF0gfCBOb25lOgogICAgICAgICIiIlBpY2sgdGhlIG5lYXJlc3QgdW50cmllZCBub24tYXZhdGFyIG9iamVjdCBjZW50cm9pZCB0byBuYXZpZ2F0ZSB0by4iIiIKICAgICAgICBvYmpzID0gUC5jb25uZWN0ZWRfY29tcG9uZW50cyhncmlkLCBiYWNrZ3JvdW5kPWJnKQogICAgICAgIGFjID0gbW0uYXZhdGFyX2NlbnRyb2lkKGdyaWQpCiAgICAgICAgaWYgYWMgaXMgTm9uZToKICAgICAgICAgICAgcmV0dXJuIE5vbmUKICAgICAgICBjYW5kcyA9IFtdCiAgICAgICAgZm9yIG8gaW4gb2JqczoKICAgICAgICAgICAgaWYgby5jb2xvciA9PSBtbS5hdmF0YXJfY29sb3I6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICByLCBjID0gaW50KHJvdW5kKG8uY2VudHJvaWRbMF0pKSwgaW50KHJvdW5kKG8uY2VudHJvaWRbMV0pKQogICAgICAgICAgICBpZiAociwgYykgaW4gdHJpZWQ6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBkID0gYWJzKHIgLSBhY1swXSkgKyBhYnMoYyAtIGFjWzFdKQogICAgICAgICAgICBjYW5kcy5hcHBlbmQoKGQsIChyLCBjKSkpCiAgICAgICAgaWYgbm90IGNhbmRzOgogICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgIGNhbmRzLnNvcnQoKQogICAgICAgIHJldHVybiBjYW5kc1swXVsxXQoKICAgIGRlZiBfbmF2aWdhdGUoc2VsZiwgZW52LCBtbTogTVYuTW90aW9uTW9kZWwsIHRhcmdldCwgcmVjb3JkX2ZuLCBzdGFydF9sZXZlbHMpOgogICAgICAgICIiIkdyZWVkaWx5IGRyaXZlIHRoZSBhdmF0YXIgdG93YXJkIHRhcmdldCB1c2luZyBsZWFybmVkIGRlbHRhcy4gUmV0dXJucyBsYXN0IG9icy4iIiIKICAgICAgICBvYnMgPSBzZWxmLl9sYXN0X29icwogICAgICAgIHRyLCB0YyA9IHRhcmdldAogICAgICAgIHN0ZXBzID0gMAogICAgICAgIHN0YWxlID0gMAogICAgICAgIHdoaWxlIHN0ZXBzIDwgc2VsZi5uYXZfc3RlcF9jYXA6CiAgICAgICAgICAgIGdyaWQgPSBQLnRvX2dyaWQob2JzLmZyYW1lKQogICAgICAgICAgICBhYyA9IG1tLmF2YXRhcl9jZW50cm9pZChncmlkKQogICAgICAgICAgICBpZiBhYyBpcyBOb25lOgogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgY3IsIGNjID0gYWMKICAgICAgICAgICAgaWYgYWJzKGNyIC0gdHIpIDwgMSBhbmQgYWJzKGNjIC0gdGMpIDwgMToKICAgICAgICAgICAgICAgIGJyZWFrICAjIGFycml2ZWQKICAgICAgICAgICAgIyBjaG9vc2UgYWN0aW9uIG1pbmltaXppbmcgcG9zdC1tb3ZlIGRpc3RhbmNlCiAgICAgICAgICAgIGJlc3RfYSwgYmVzdF9kID0gTm9uZSwgTm9uZQogICAgICAgICAgICBjdXJfZCA9IGFicyhjciAtIHRyKSArIGFicyhjYyAtIHRjKQogICAgICAgICAgICBmb3IgYWlkLCAoZHIsIGRjKSBpbiBtbS5kZWx0YXMuaXRlbXMoKToKICAgICAgICAgICAgICAgIG5kID0gYWJzKGNyICsgZHIgLSB0cikgKyBhYnMoY2MgKyBkYyAtIHRjKQogICAgICAgICAgICAgICAgaWYgYmVzdF9kIGlzIE5vbmUgb3IgbmQgPCBiZXN0X2Q6CiAgICAgICAgICAgICAgICAgICAgYmVzdF9kLCBiZXN0X2EgPSBuZCwgYWlkCiAgICAgICAgICAgIGlmIGJlc3RfYSBpcyBOb25lIG9yIGJlc3RfZCA+PSBjdXJfZDoKICAgICAgICAgICAgICAgIGJyZWFrICAjIG5vIGltcHJvdmluZyBtb3ZlIChncmVlZHkgc3R1Y2spCiAgICAgICAgICAgIGFjdGlvbiA9ICgiUyIsIGJlc3RfYSkKICAgICAgICAgICAgZ2EsIF8gPSB0b19nYW1lX2FjdGlvbihhY3Rpb24pCiAgICAgICAgICAgIGJlZm9yZV9sZXZlbHMgPSBpbnQob2JzLmxldmVsc19jb21wbGV0ZWQgb3IgMCkKICAgICAgICAgICAgbyA9IGVudi5zdGVwKGdhKQogICAgICAgICAgICByZWNvcmRfZm4oYWN0aW9uLCBvKQogICAgICAgICAgICBvYnMgPSBvCiAgICAgICAgICAgIHNlbGYuX2xhc3Rfb2JzID0gbwogICAgICAgICAgICBzdGVwcyArPSAxCiAgICAgICAgICAgIGlmIG8uc3RhdGUgaW4gKEdhbWVTdGF0ZS5XSU4sIEdhbWVTdGF0ZS5HQU1FX09WRVIpOgogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgaWYgaW50KG8ubGV2ZWxzX2NvbXBsZXRlZCBvciAwKSA+IGJlZm9yZV9sZXZlbHM6CiAgICAgICAgICAgICAgICBicmVhayAgIyByZXdhcmQhCiAgICAgICAgICAgICMgZGV0ZWN0IGJsb2NrZWQgKGF2YXRhciBkaWRuJ3QgbW92ZSkgLT4gc3RvcCB0byBhdm9pZCBzcGluCiAgICAgICAgICAgIG5nID0gUC50b19ncmlkKG8uZnJhbWUpCiAgICAgICAgICAgIG5hYyA9IG1tLmF2YXRhcl9jZW50cm9pZChuZykKICAgICAgICAgICAgaWYgbmFjIGlzIG5vdCBOb25lIGFuZCBhYnMobmFjWzBdIC0gY3IpIDwgMC41IGFuZCBhYnMobmFjWzFdIC0gY2MpIDwgMC41OgogICAgICAgICAgICAgICAgc3RhbGUgKz0gMQogICAgICAgICAgICAgICAgaWYgc3RhbGUgPj0gMjoKICAgICAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgc3RhbGUgPSAwCiAgICAgICAgcmV0dXJuIG9icwo=', 'policy.py': 'IiIiUmVhY3RpdmUgaHlicmlkIHBvbGljeTogb25lIGFjdGlvbiBwZXIgY2FsbCwgc3RhdGUgcGVyc2lzdGVkIG9uIHRoZSBvYmplY3QuCgpUaGlzIGlzIHRoZSBzdWJtaXNzaW9uLXNoYXBlZCBmb3JtIG9mIHRoZSBhZ2VudC4gVGhlIEthZ2dsZSBldmFsIGRyaXZlcyBhZ2VudHMgdmlhIHRoZQpvZmZpY2lhbCBgQWdlbnQuY2hvb3NlX2FjdGlvbihmcmFtZXMsIGxhdGVzdF9mcmFtZSkgLT4gR2FtZUFjdGlvbmAgaW50ZXJmYWNlIChvbmUgYWN0aW9uCmF0IGEgdGltZSwgcmVzdWx0IG9ic2VydmVkIG9uIHRoZSBuZXh0IGNhbGwpLiBIeWJyaWRQb2xpY3kgaW1wbGVtZW50cyB0aGUgc2FtZSBzdHJhdGVneQphcyBIeWJyaWRBZ2VudCAobW90aW9uIG1vZGVsICsgY29vcmRpbmF0ZSBuYXZpZ2F0aW9uICsgZ3JhcGgtZXhwbG9yYXRpb24gZmFsbGJhY2spIGJ1dCBhcwphbiBpbmNyZW1lbnRhbCBzdGF0ZSBtYWNoaW5lLCBzbyBpdCB3b3JrcyBib3RoIHRocm91Z2ggdGhlIG9mZmljaWFsIGZyYW1ld29yayBhbmQgdGhyb3VnaApvdXIgb3duIHJlYWN0aXZlIHJ1bm5lciAvIG9mZmxpbmUgZW52LgoKQWN0aW9uIHRva2VuczogKCJyZXNldCIsKSB8ICgiUyIsIGlkKSB8ICgiQyIsIHgsIHkpIOKAlCB0aGUgY2FsbGVyIG1hcHMgdGhlc2UgdG8gR2FtZUFjdGlvbi4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbnVtcHkgYXMgbnAKCmZyb20gLiBpbXBvcnQgbW92ZW1lbnQgYXMgTVYKZnJvbSAuIGltcG9ydCBwZXJjZXB0aW9uIGFzIFAKZnJvbSAuYWdlbnQgaW1wb3J0IEFDVCwgUkVTRVQsIFNUT1AsIEdyYXBoU3RyYXRlZ3ksIGNhbmRpZGF0ZXNfZm9yCmZyb20gLndvcmxkX21vZGVsIGltcG9ydCBBY3Rpb24KCgpjbGFzcyBIeWJyaWRQb2xpY3k6CiAgICBkZWYgX19pbml0X18oc2VsZiwgdXNlX2NsaWNrczogYm9vbCA9IFRydWUsIG1heF9jbGlja190YXJnZXRzOiBpbnQgPSA5NiwKICAgICAgICAgICAgICAgICBuYXZfc3RlcF9jYXA6IGludCA9IDIwMCwgc2VlZDogaW50ID0gMCkgLT4gTm9uZToKICAgICAgICBzZWxmLnVzZV9jbGlja3MgPSB1c2VfY2xpY2tzCiAgICAgICAgc2VsZi5tYXhfY2xpY2tfdGFyZ2V0cyA9IG1heF9jbGlja190YXJnZXRzCiAgICAgICAgc2VsZi5uYXZfc3RlcF9jYXAgPSBuYXZfc3RlcF9jYXAKICAgICAgICBzZWxmLnJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhzZWVkKQogICAgICAgIHNlbGYucmVzZXRfYWxsKCkKCiAgICBkZWYgcmVzZXRfYWxsKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgc2VsZi52dCA9IFAuVm9sYXRpbGl0eVRyYWNrZXIoKQogICAgICAgIHNlbGYucm9vdF9rZXk6IGJ5dGVzIHwgTm9uZSA9IE5vbmUKICAgICAgICBzZWxmLmdzOiBHcmFwaFN0cmF0ZWd5IHwgTm9uZSA9IE5vbmUKICAgICAgICBzZWxmLmJnOiBpbnQgfCBOb25lID0gTm9uZQogICAgICAgIHNlbGYucHJldl9rZXk6IGJ5dGVzIHwgTm9uZSA9IE5vbmUKICAgICAgICBzZWxmLnByZXZfYWN0aW9uOiBBY3Rpb24gfCBOb25lID0gTm9uZQogICAgICAgIHNlbGYucHJldl9sZXZlbHMgPSAwCiAgICAgICAgc2VsZi5sZXZlbCA9IC0xCiAgICAgICAgc2VsZi5leHBlY3RfcmVzZXQgPSBGYWxzZQogICAgICAgICMgbW90aW9uIC8gcGhhc2UKICAgICAgICBzZWxmLnBoYXNlID0gInByb2JlIgogICAgICAgIHNlbGYubW06IE1WLk1vdGlvbk1vZGVsIHwgTm9uZSA9IE5vbmUKICAgICAgICBzZWxmLl92b3RlczogZGljdFtpbnQsIGRpY3RbaW50LCB0dXBsZVtpbnQsIGludF1dXSA9IHt9CiAgICAgICAgc2VsZi5fY2hhbmdlZF9jb2xvcnM6IHNldFtpbnRdID0gc2V0KCkKICAgICAgICBzZWxmLmRpc3RyYWN0b3JfY29sb3JzOiBzZXRbaW50XSA9IHNldCgpICAjIGFuaW1hdGVkL2NvdW50ZXIgY29sb3JzIHRvIG1hc2sgKyBpZ25vcmUKICAgICAgICBzZWxmLl9wcm9iZV9xdWV1ZTogbGlzdFtpbnRdIHwgTm9uZSA9IE5vbmUKICAgICAgICBzZWxmLl9wcm9iZV9iZWZvcmU6IG5wLm5kYXJyYXkgfCBOb25lID0gTm9uZQogICAgICAgIHNlbGYuX3Byb2JlX2FpZDogaW50IHwgTm9uZSA9IE5vbmUKICAgICAgICAjIG5hdmlnYXRpb24KICAgICAgICBzZWxmLnRhcmdldDogdHVwbGVbaW50LCBpbnRdIHwgTm9uZSA9IE5vbmUKICAgICAgICBzZWxmLnRyaWVkX3RhcmdldHM6IHNldFt0dXBsZVtpbnQsIGludF1dID0gc2V0KCkKICAgICAgICBzZWxmLm5hdl9zdGVwcyA9IDAKICAgICAgICBzZWxmLm5hdl9zdGFsZSA9IDAKICAgICAgICBzZWxmLm5hdl9sYXN0OiB0dXBsZVtmbG9hdCwgZmxvYXRdIHwgTm9uZSA9IE5vbmUKCiAgICBkZWYgX2NhbmRzKHNlbGYsIGdyaWQsIGF2YWlsYWJsZSk6CiAgICAgICAgcmV0dXJuIGNhbmRpZGF0ZXNfZm9yKGdyaWQsIGF2YWlsYWJsZSwgc2VsZi51c2VfY2xpY2tzLCBzZWxmLm1heF9jbGlja190YXJnZXRzLCBGYWxzZSkKCiAgICBkZWYgX2tleShzZWxmLCBncmlkOiBucC5uZGFycmF5KSAtPiBieXRlczoKICAgICAgICAiIiJPYmplY3Qtc3RydWN0dXJlIHN0YXRlIGtleSAocm9idXN0IHRvIHBpeGVsIG5vaXNlKSwgaWdub3JpbmcgYW5pbWF0ZWQgZGlzdHJhY3RvcnMuCgogICAgICAgIE9iamVjdC1sZXZlbCBoYXNoaW5nIGNvbGxhcHNlcyBpcnJlbGV2YW50IHBlci1waXhlbCBqaXR0ZXIgdGhhdCB3b3VsZCBvdGhlcndpc2UKICAgICAgICBleHBsb2RlIHRoZSBzdGF0ZSBncmFwaCBvbiByZWFsIGdhbWVzOyBhbmltYXRlZC1kaXN0cmFjdG9yIGNvbG9ycyBhcmUgZXhjbHVkZWQuCiAgICAgICAgIiIiCiAgICAgICAgcmV0dXJuIFAub2JqZWN0X3N0YXRlX2tleShncmlkLCBiYWNrZ3JvdW5kPXNlbGYuYmcsIGlnbm9yZV9jb2xvcnM9c2VsZi5kaXN0cmFjdG9yX2NvbG9ycykKCiAgICBkZWYgX25ld19sZXZlbChzZWxmLCBsZXZlbHM6IGludCkgLT4gTm9uZToKICAgICAgICBzZWxmLmxldmVsID0gbGV2ZWxzCiAgICAgICAgc2VsZi5waGFzZSA9ICJwcm9iZSIKICAgICAgICBzZWxmLm1tID0gTm9uZQogICAgICAgIHNlbGYuX3ZvdGVzID0ge30KICAgICAgICBzZWxmLl9jaGFuZ2VkX2NvbG9ycyA9IHNldCgpCiAgICAgICAgc2VsZi5kaXN0cmFjdG9yX2NvbG9ycyA9IHNldCgpCiAgICAgICAgc2VsZi5iZyA9IE5vbmUKICAgICAgICBzZWxmLl9wcm9iZV9xdWV1ZSA9IE5vbmUKICAgICAgICBzZWxmLl9wcm9iZV9iZWZvcmUgPSBOb25lCiAgICAgICAgc2VsZi5fcHJvYmVfYWlkID0gTm9uZQogICAgICAgIHNlbGYudGFyZ2V0ID0gTm9uZQogICAgICAgIHNlbGYudHJpZWRfdGFyZ2V0cyA9IHNldCgpCgogICAgIyBtYWluIGVudHJ5OiBnaXZlbiB0aGUgbGF0ZXN0IG9ic2VydmF0aW9uLCByZXR1cm4gdGhlIG5leHQgYWN0aW9uIHRva2VuCiAgICBkZWYgZGVjaWRlKHNlbGYsIGdyaWQ6IG5wLm5kYXJyYXksIGdzdGF0ZV90ZXJtaW5hbDogYm9vbCwgZ3N0YXRlX25vdHBsYXllZDogYm9vbCwKICAgICAgICAgICAgICAgbGV2ZWxzOiBpbnQsIGF2YWlsYWJsZTogbGlzdFtpbnRdKSAtPiBBY3Rpb246CiAgICAgICAgc2VsZi52dC51cGRhdGUoZ3JpZCkKICAgICAgICBpZiBzZWxmLmJnIGlzIE5vbmU6CiAgICAgICAgICAgIHNlbGYuYmcgPSBQLmRldGVjdF9iYWNrZ3JvdW5kKGdyaWQpCiAgICAgICAgY3VyX2tleSA9IHNlbGYuX2tleShncmlkKQoKICAgICAgICAjIHRlcm1pbmFsIC8gbm90LXBsYXllZCAtPiBSRVNFVAogICAgICAgIGlmIGdzdGF0ZV90ZXJtaW5hbCBvciBnc3RhdGVfbm90cGxheWVkOgogICAgICAgICAgICBpZiBnc3RhdGVfdGVybWluYWwgYW5kIHNlbGYucHJldl9hY3Rpb24gaXMgbm90IE5vbmUgYW5kIHNlbGYucHJldl9rZXkgaXMgbm90IE5vbmUgYW5kIHNlbGYuZ3M6CiAgICAgICAgICAgICAgICBzZWxmLmdzLnVwZGF0ZShzZWxmLnByZXZfa2V5LCBzZWxmLnByZXZfYWN0aW9uLCBjdXJfa2V5LCAwLjAsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzZWxmLl9jYW5kcyhncmlkLCBhdmFpbGFibGUpLCB0ZXJtaW5hbD1UcnVlKQogICAgICAgICAgICBzZWxmLnByZXZfYWN0aW9uID0gTm9uZQogICAgICAgICAgICBzZWxmLmV4cGVjdF9yZXNldCA9IFRydWUKICAgICAgICAgICAgaWYgc2VsZi5nczoKICAgICAgICAgICAgICAgIHNlbGYuZ3MucGxhbiA9IFtdCiAgICAgICAgICAgIHJldHVybiAoInJlc2V0IiwpCgogICAgICAgICMgZmlyc3QgcmVhbCBmcmFtZSAoYWZ0ZXIgaW5pdGlhbCByZXNldCkgLT4gZXN0YWJsaXNoIHJvb3QKICAgICAgICBpZiBzZWxmLnJvb3Rfa2V5IGlzIE5vbmU6CiAgICAgICAgICAgIHNlbGYucm9vdF9rZXkgPSBjdXJfa2V5CiAgICAgICAgICAgIHNlbGYuZ3MgPSBHcmFwaFN0cmF0ZWd5KHNlbGYucm9vdF9rZXkpCiAgICAgICAgICAgIHNlbGYuZ3Mud20ub2JzZXJ2ZShzZWxmLnJvb3Rfa2V5LCBzZWxmLl9jYW5kcyhncmlkLCBhdmFpbGFibGUpKQogICAgICAgICAgICBzZWxmLl9uZXdfbGV2ZWwobGV2ZWxzKQogICAgICAgICAgICBzZWxmLmJnID0gUC5kZXRlY3RfYmFja2dyb3VuZChncmlkKQoKICAgICAgICBpZiBzZWxmLmV4cGVjdF9yZXNldDoKICAgICAgICAgICAgc2VsZi5leHBlY3RfcmVzZXQgPSBGYWxzZQogICAgICAgICAgICBzZWxmLnByZXZfYWN0aW9uID0gTm9uZSAgIyBkb24ndCByZWNvcmQgYWNyb3NzIHJlc2V0CgogICAgICAgICMgcmVjb3JkIG91dGNvbWUgb2YgdGhlIHByZXZpb3VzIGFjdGlvbgogICAgICAgIGlmIHNlbGYucHJldl9hY3Rpb24gaXMgbm90IE5vbmUgYW5kIHNlbGYucHJldl9rZXkgaXMgbm90IE5vbmUgYW5kIHNlbGYuZ3MgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHJld2FyZCA9IGZsb2F0KGxldmVscyAtIHNlbGYucHJldl9sZXZlbHMpCiAgICAgICAgICAgIHNlbGYuZ3MudXBkYXRlKHNlbGYucHJldl9rZXksIHNlbGYucHJldl9hY3Rpb24sIGN1cl9rZXksIHJld2FyZCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgc2VsZi5fY2FuZHMoZ3JpZCwgYXZhaWxhYmxlKSwgdGVybWluYWw9RmFsc2UpCiAgICAgICAgICAgIGlmIHNlbGYucGhhc2UgPT0gInByb2JlIiBhbmQgc2VsZi5fcHJvYmVfYmVmb3JlIGlzIG5vdCBOb25lIGFuZCBzZWxmLl9wcm9iZV9haWQgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICB0cmFucyA9IE1WLmluZmVyX2FsbF90cmFuc2xhdGlvbnMoc2VsZi5fcHJvYmVfYmVmb3JlLCBncmlkLCBzZWxmLmJnKQogICAgICAgICAgICAgICAgZm9yIGNvbG9yLCAoZHIsIGRjKSBpbiB0cmFucy5pdGVtcygpOgogICAgICAgICAgICAgICAgICAgIHNlbGYuX3ZvdGVzLnNldGRlZmF1bHQoY29sb3IsIHt9KVtzZWxmLl9wcm9iZV9haWRdID0gKGRyLCBkYykKICAgICAgICAgICAgICAgICMgYW55IG5vbi1iYWNrZ3JvdW5kIGNvbG9yIHdob3NlIGNlbGxzIGNoYW5nZWQgdGhpcyBzdGVwCiAgICAgICAgICAgICAgICBmb3IgYyBpbiBzZXQobnAudW5pcXVlKHNlbGYuX3Byb2JlX2JlZm9yZSkpLnVuaW9uKG5wLnVuaXF1ZShncmlkKSk6CiAgICAgICAgICAgICAgICAgICAgYyA9IGludChjKQogICAgICAgICAgICAgICAgICAgIGlmIGMgIT0gc2VsZi5iZyBhbmQgbm90IG5wLmFycmF5X2VxdWFsKHNlbGYuX3Byb2JlX2JlZm9yZSA9PSBjLCBncmlkID09IGMpOgogICAgICAgICAgICAgICAgICAgICAgICBzZWxmLl9jaGFuZ2VkX2NvbG9ycy5hZGQoYykKCiAgICAgICAgIyBuZXcgbGV2ZWwgLT4gcmVsZWFybgogICAgICAgIGlmIGxldmVscyAhPSBzZWxmLmxldmVsOgogICAgICAgICAgICBzZWxmLl9uZXdfbGV2ZWwobGV2ZWxzKQoKICAgICAgICBzZWxmLnByZXZfbGV2ZWxzID0gbGV2ZWxzCiAgICAgICAgYWN0aW9uID0gc2VsZi5fY2hvb3NlKGdyaWQsIGN1cl9rZXksIGF2YWlsYWJsZSkKICAgICAgICBzZWxmLnByZXZfa2V5ID0gY3VyX2tleQogICAgICAgIHNlbGYucHJldl9hY3Rpb24gPSBOb25lIGlmIGFjdGlvblswXSA9PSAicmVzZXQiIGVsc2UgYWN0aW9uCiAgICAgICAgcmV0dXJuIGFjdGlvbgoKICAgIGRlZiBfY2hvb3NlKHNlbGYsIGdyaWQsIGN1cl9rZXksIGF2YWlsYWJsZSwgX2RlcHRoOiBpbnQgPSAwKSAtPiBBY3Rpb246CiAgICAgICAgaWYgX2RlcHRoID4gMzoKICAgICAgICAgICAgcmV0dXJuIHNlbGYuX3JhbmRvbV9hY3Rpb24oZ3JpZCwgYXZhaWxhYmxlKQogICAgICAgIHNpbXBsZV9hdmFpbCA9IFthIGZvciBhIGluICgxLCAyLCAzLCA0LCA1KSBpZiBhIGluIGF2YWlsYWJsZV0KCiAgICAgICAgIyAtLS0tLSBQUk9CRTogbGVhcm4gbW90aW9uIG1vZGVsIC0tLS0tCiAgICAgICAgaWYgc2VsZi5waGFzZSA9PSAicHJvYmUiOgogICAgICAgICAgICBpZiBzZWxmLl9wcm9iZV9xdWV1ZSBpcyBOb25lOgogICAgICAgICAgICAgICAgc2VsZi5fcHJvYmVfcXVldWUgPSBsaXN0KHNpbXBsZV9hdmFpbCkKICAgICAgICAgICAgaWYgc2VsZi5fcHJvYmVfcXVldWU6CiAgICAgICAgICAgICAgICBhaWQgPSBzZWxmLl9wcm9iZV9xdWV1ZS5wb3AoMCkKICAgICAgICAgICAgICAgIHNlbGYuX3Byb2JlX2JlZm9yZSA9IGdyaWQKICAgICAgICAgICAgICAgIHNlbGYuX3Byb2JlX2FpZCA9IGFpZAogICAgICAgICAgICAgICAgcmV0dXJuICgiUyIsIGFpZCkKICAgICAgICAgICAgIyBmaW5pc2hlZCBwcm9iaW5nOiB0aGUgYXZhdGFyIGlzIHRoZSBvYmplY3Qgd2hvc2UgbW90aW9uIENPUlJFTEFURVMgd2l0aCB0aGUKICAgICAgICAgICAgIyBhY3Rpb24gKG1vc3QgZGlzdGluY3QgZGVsdGEgdmVjdG9ycyk7IGNvdW50ZXJzL2FuaW1hdGlvbnMgbW92ZSBjb25zdGFudGx5LgogICAgICAgICAgICBpZiBzZWxmLl92b3RlczoKICAgICAgICAgICAgICAgIGRlZiBfc2NvcmUoYyk6CiAgICAgICAgICAgICAgICAgICAgZGVsdGFzID0gc2VsZi5fdm90ZXNbY10KICAgICAgICAgICAgICAgICAgICByZXR1cm4gKGxlbihzZXQoZGVsdGFzLnZhbHVlcygpKSksIGxlbihkZWx0YXMpKQogICAgICAgICAgICAgICAgY29sb3IgPSBtYXgoc2VsZi5fdm90ZXMsIGtleT1fc2NvcmUpCiAgICAgICAgICAgICAgICAjIHRoZSBhdmF0YXIgbWF5IHNwYW4gTVVMVElQTEUgY29sb3JzIHRoYXQgbW92ZSB0b2dldGhlciAoYWN0aW9uLWNvcnJlbGF0ZWQsCiAgICAgICAgICAgICAgICAjIGkuZS4gPj0yIGRpc3RpbmN0IGRlbHRhcyk7IHRyYWNrIHRoZW0gYWxsIGZvciBjZW50cm9pZCArIHRhcmdldCBleGNsdXNpb24uCiAgICAgICAgICAgICAgICBhdmF0YXJfY29sb3JzID0gZnJvemVuc2V0KAogICAgICAgICAgICAgICAgICAgIGMgZm9yIGMsIGQgaW4gc2VsZi5fdm90ZXMuaXRlbXMoKSBpZiBsZW4oc2V0KGQudmFsdWVzKCkpKSA+PSAyCiAgICAgICAgICAgICAgICApIG9yIGZyb3plbnNldCh7Y29sb3J9KQogICAgICAgICAgICAgICAgc2VsZi5tbSA9IE1WLk1vdGlvbk1vZGVsKGF2YXRhcl9jb2xvcj1jb2xvciwgZGVsdGFzPXNlbGYuX3ZvdGVzW2NvbG9yXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBhdmF0YXJfY29sb3JzPWF2YXRhcl9jb2xvcnMpCiAgICAgICAgICAgICAgICAjIEFuaW1hdGVkIGRpc3RyYWN0b3IgPSBhIGNvbG9yIHRoYXQgUklHSURMWSBUUkFOU0xBVEVTIHdpdGggYSBjb25zdGFudAogICAgICAgICAgICAgICAgIyBkZWx0YSByZWdhcmRsZXNzIG9mIHRoZSBhY3Rpb24gKGEgY291bnRlci9hbmltYXRpb24pLCBOT1QgbWVyZWx5IGEgY29sb3IKICAgICAgICAgICAgICAgICMgd2hvc2UgY2VsbHMgY2hhbmdlZCAodGhhdCBhbHNvIGZsYWdzIHN0cnVjdHVyYWwgY2VsbHMgdGhlIGF2YXRhciBtb3ZlcwogICAgICAgICAgICAgICAgIyBvdmVyLCBlLmcuIG1hemUgd2FsbHMg4oCUIHdoaWNoIHdvdWxkIGJsaW5kIHVzIHRvIGRvb3JzIG9wZW5pbmcpLgogICAgICAgICAgICAgICAgc2VsZi5kaXN0cmFjdG9yX2NvbG9ycyA9IHsKICAgICAgICAgICAgICAgICAgICBjIGZvciBjLCBkIGluIHNlbGYuX3ZvdGVzLml0ZW1zKCkKICAgICAgICAgICAgICAgICAgICBpZiBjICE9IGNvbG9yIGFuZCBjICE9IHNlbGYuYmcKICAgICAgICAgICAgICAgICAgICBhbmQgbGVuKGQpID49IDIgYW5kIGxlbihzZXQoZC52YWx1ZXMoKSkpID09IDEKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgICAgIHNlbGYucGhhc2UgPSAibmF2aWdhdGUiCiAgICAgICAgICAgICAgICBzZWxmLnRhcmdldCA9IE5vbmUKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHNlbGYucGhhc2UgPSAiZ3JhcGgiCiAgICAgICAgICAgIHJldHVybiBzZWxmLl9jaG9vc2UoZ3JpZCwgY3VyX2tleSwgYXZhaWxhYmxlLCBfZGVwdGggKyAxKQoKICAgICAgICAjIC0tLS0tIE5BVklHQVRFIGF2YXRhciB0byBjYW5kaWRhdGUgZ29hbCBvYmplY3RzIC0tLS0tCiAgICAgICAgaWYgc2VsZi5waGFzZSA9PSAibmF2aWdhdGUiIGFuZCBzZWxmLm1tIGlzIG5vdCBOb25lIGFuZCBzZWxmLm1tLm9rOgogICAgICAgICAgICBpZiBzZWxmLnRhcmdldCBpcyBOb25lOgogICAgICAgICAgICAgICAgdCA9IHNlbGYuX25leHRfdGFyZ2V0KGdyaWQpCiAgICAgICAgICAgICAgICBpZiB0IGlzIE5vbmU6CiAgICAgICAgICAgICAgICAgICAgc2VsZi5waGFzZSA9ICJncmFwaCIKICAgICAgICAgICAgICAgICAgICByZXR1cm4gc2VsZi5fY2hvb3NlKGdyaWQsIGN1cl9rZXksIGF2YWlsYWJsZSwgX2RlcHRoICsgMSkKICAgICAgICAgICAgICAgIHNlbGYudGFyZ2V0ID0gdAogICAgICAgICAgICAgICAgc2VsZi50cmllZF90YXJnZXRzLmFkZCh0KQogICAgICAgICAgICAgICAgc2VsZi5uYXZfc3RlcHMgPSAwCiAgICAgICAgICAgICAgICBzZWxmLm5hdl9zdGFsZSA9IDAKICAgICAgICAgICAgICAgIHNlbGYubmF2X2xhc3QgPSBOb25lCiAgICAgICAgICAgIGFjdCA9IHNlbGYuX25hdl9zdGVwKGdyaWQpCiAgICAgICAgICAgIGlmIGFjdCBpcyBOb25lOgogICAgICAgICAgICAgICAgc2VsZi50YXJnZXQgPSBOb25lCiAgICAgICAgICAgICAgICByZXR1cm4gc2VsZi5fY2hvb3NlKGdyaWQsIGN1cl9rZXksIGF2YWlsYWJsZSwgX2RlcHRoICsgMSkKICAgICAgICAgICAgcmV0dXJuIGFjdAoKICAgICAgICAjIC0tLS0tIEdSQVBIIGZhbGxiYWNrIC0tLS0tCiAgICAgICAgaWYgc2VsZi5ncyBpcyBub3QgTm9uZToKICAgICAgICAgICAga2luZCwgYSA9IHNlbGYuZ3MuZGVjaWRlKGN1cl9rZXkpCiAgICAgICAgICAgIGlmIGtpbmQgPT0gUkVTRVQ6CiAgICAgICAgICAgICAgICBzZWxmLmV4cGVjdF9yZXNldCA9IFRydWUKICAgICAgICAgICAgICAgIHNlbGYuZ3MucGxhbiA9IFtdCiAgICAgICAgICAgICAgICByZXR1cm4gKCJyZXNldCIsKQogICAgICAgICAgICBpZiBraW5kID09IFNUT1A6CiAgICAgICAgICAgICAgICByZXR1cm4gc2VsZi5fcmFuZG9tX2FjdGlvbihncmlkLCBhdmFpbGFibGUpCiAgICAgICAgICAgIHJldHVybiBhCiAgICAgICAgcmV0dXJuIHNlbGYuX3JhbmRvbV9hY3Rpb24oZ3JpZCwgYXZhaWxhYmxlKQoKICAgIGRlZiBfbmV4dF90YXJnZXQoc2VsZiwgZ3JpZCk6CiAgICAgICAgb2JqcyA9IFAuY29ubmVjdGVkX2NvbXBvbmVudHMoZ3JpZCwgYmFja2dyb3VuZD1zZWxmLmJnKQogICAgICAgIGFjID0gc2VsZi5tbS5hdmF0YXJfY2VudHJvaWQoZ3JpZCkKICAgICAgICBpZiBhYyBpcyBOb25lOgogICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgIGNhbmRzID0gW10KICAgICAgICBmb3IgbyBpbiBvYmpzOgogICAgICAgICAgICBpZiBvLmNvbG9yIGluIHNlbGYubW0uYXZhdGFyX2NvbG9ycyBvciBvLmNvbG9yIGluIHNlbGYuZGlzdHJhY3Rvcl9jb2xvcnM6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICByLCBjID0gaW50KHJvdW5kKG8uY2VudHJvaWRbMF0pKSwgaW50KHJvdW5kKG8uY2VudHJvaWRbMV0pKQogICAgICAgICAgICBpZiAociwgYykgaW4gc2VsZi50cmllZF90YXJnZXRzOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgY2FuZHMuYXBwZW5kKChhYnMociAtIGFjWzBdKSArIGFicyhjIC0gYWNbMV0pLCAociwgYykpKQogICAgICAgIGlmIG5vdCBjYW5kczoKICAgICAgICAgICAgcmV0dXJuIE5vbmUKICAgICAgICBjYW5kcy5zb3J0KCkKICAgICAgICByZXR1cm4gY2FuZHNbMF1bMV0KCiAgICBkZWYgX25hdl9zdGVwKHNlbGYsIGdyaWQpOgogICAgICAgIGFjID0gc2VsZi5tbS5hdmF0YXJfY2VudHJvaWQoZ3JpZCkKICAgICAgICBpZiBhYyBpcyBOb25lOgogICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgIGNyLCBjYyA9IGFjCiAgICAgICAgdHIsIHRjID0gc2VsZi50YXJnZXQKICAgICAgICBpZiBhYnMoY3IgLSB0cikgPCAxIGFuZCBhYnMoY2MgLSB0YykgPCAxOgogICAgICAgICAgICByZXR1cm4gTm9uZSAgIyBhcnJpdmVkCiAgICAgICAgaWYgc2VsZi5uYXZfc3RlcHMgPj0gc2VsZi5uYXZfc3RlcF9jYXA6CiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgIyBkZXRlY3Qgc3RhbGxlZCBhdmF0YXIgKGRpZG4ndCBtb3ZlIHNpbmNlIGxhc3QgbmF2IGFjdGlvbikKICAgICAgICBpZiBzZWxmLm5hdl9sYXN0IGlzIG5vdCBOb25lIGFuZCBhYnMoc2VsZi5uYXZfbGFzdFswXSAtIGNyKSA8IDAuNSBhbmQgYWJzKHNlbGYubmF2X2xhc3RbMV0gLSBjYykgPCAwLjU6CiAgICAgICAgICAgIHNlbGYubmF2X3N0YWxlICs9IDEKICAgICAgICAgICAgaWYgc2VsZi5uYXZfc3RhbGUgPj0gMjoKICAgICAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc2VsZi5uYXZfc3RhbGUgPSAwCiAgICAgICAgY3VyX2QgPSBhYnMoY3IgLSB0cikgKyBhYnMoY2MgLSB0YykKICAgICAgICBiZXN0X2EsIGJlc3RfZCA9IE5vbmUsIE5vbmUKICAgICAgICBmb3IgYWlkLCAoZHIsIGRjKSBpbiBzZWxmLm1tLmRlbHRhcy5pdGVtcygpOgogICAgICAgICAgICBuZCA9IGFicyhjciArIGRyIC0gdHIpICsgYWJzKGNjICsgZGMgLSB0YykKICAgICAgICAgICAgaWYgYmVzdF9kIGlzIE5vbmUgb3IgbmQgPCBiZXN0X2Q6CiAgICAgICAgICAgICAgICBiZXN0X2QsIGJlc3RfYSA9IG5kLCBhaWQKICAgICAgICBpZiBiZXN0X2EgaXMgTm9uZSBvciBiZXN0X2QgPj0gY3VyX2Q6CiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgc2VsZi5uYXZfbGFzdCA9IChjciwgY2MpCiAgICAgICAgc2VsZi5uYXZfc3RlcHMgKz0gMQogICAgICAgIHJldHVybiAoIlMiLCBiZXN0X2EpCgogICAgZGVmIF9yYW5kb21fYWN0aW9uKHNlbGYsIGdyaWQsIGF2YWlsYWJsZSkgLT4gQWN0aW9uOgogICAgICAgIGNhbmRzID0gc2VsZi5fY2FuZHMoZ3JpZCwgYXZhaWxhYmxlKQogICAgICAgIGlmIG5vdCBjYW5kczoKICAgICAgICAgICAgcmV0dXJuICgiUyIsIGF2YWlsYWJsZVswXSkgaWYgYXZhaWxhYmxlIGVsc2UgKCJyZXNldCIsKQogICAgICAgIGkgPSBpbnQoc2VsZi5ybmcuaW50ZWdlcnMoMCwgbGVuKGNhbmRzKSkpCiAgICAgICAgcmV0dXJuIGNhbmRzW2ldCg==', 'spatial.py': 'IiIiU3BhdGlhbCBzY2VuZSBtb2RlbDogbGVhcm4gYW4gb2NjdXBhbmN5IG1hcCBvZiB0aGUgYXZhdGFyJ3Mgd29ybGQgYW5kIEEqLXBhdGhmaW5kLgoKUGFydCBvZiB0aGUgc3RydWN0dXJlZCB3b3JsZC1tb2RlbCByZWJ1aWxkLiBUaGUgYWdlbnQncyBhdmF0YXIgbW92ZXMgb24gYSBsYXR0aWNlIChlYWNoCnNpbXBsZSBhY3Rpb24gc2hpZnRzIGl0cyBjZW50cm9pZCBieSBhIHJvdWdobHktY29uc3RhbnQgZGVsdGEpLiBCeSByZWNvcmRpbmcgd2hpY2ggbGF0dGljZQpwb3NpdGlvbnMgdGhlIGF2YXRhciBzdWNjZXNzZnVsbHkgZW50ZXJlZCAoZnJlZSkgdmVyc3VzIHRyaWVkLWFuZC13YXMtYmxvY2tlZCAod2FsbCksIHdlCmJ1aWxkIGFuIG9jY3VwYW5jeSBtYXAgYW5kIHBsYW4gb3B0aW1hbCBwYXRocyB0byBhbnkgdGFyZ2V0IHdpdGggQSog4oCUIGluc3RlYWQgb2YgZ3JlZWR5Cm5hdmlnYXRpb24gdGhhdCBzdGFsbHMgYXQgdGhlIGZpcnN0IG9ic3RhY2xlLiBVbmtub3duIGNlbGxzIGFyZSB0cmVhdGVkIGFzIGZyZWUKKG9wdGltaXN0aWMpLCBzbyB0aGUgcGxhbm5lciByb3V0ZXMgYXJvdW5kICprbm93biogd2FsbHMgYW5kIHByb2JlcyB0aGUgdW5rbm93bi4KCkFzc3VtZXMgYXhpcy1hbGlnbmVkIG1vdmVtZW50ICh0aGUgZG9taW5hbnQgQVJDLUFHSS0zIGNvbnRyb2wgc2NoZW1lKS4gSWYgZGVsdGFzIGFyZW4ndApheGlzLWFsaWduZWQvY29uc2lzdGVudCwgdGhlIGNhbGxlciBzaG91bGQgbm90IHVzZSB0aGlzIGFuZCBmYWxsIGJhY2sgdG8gZ3JhcGggZXhwbG9yYXRpb24uCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGhlYXBxCmZyb20gbWF0aCBpbXBvcnQgZ2NkCgoKY2xhc3MgT2NjdXBhbmN5TWFwOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGRlbHRhczogZGljdFtpbnQsIHR1cGxlW2ludCwgaW50XV0pOgogICAgICAgICMga2VlcCBvbmx5IG5vbnplcm8sIGF4aXMtYWxpZ25lZCBkZWx0YXMgKG9uZSBheGlzIHplcm8pCiAgICAgICAgc2VsZi5kZWx0YXMgPSB7CiAgICAgICAgICAgIGE6IChkciwgZGMpIGZvciBhLCAoZHIsIGRjKSBpbiBkZWx0YXMuaXRlbXMoKQogICAgICAgICAgICBpZiAoZHIsIGRjKSAhPSAoMCwgMCkgYW5kIChkciA9PSAwIG9yIGRjID09IDApCiAgICAgICAgfQogICAgICAgIG1hZ3MgPSBbYWJzKGRyKSBvciBhYnMoZGMpIGZvciBkciwgZGMgaW4gc2VsZi5kZWx0YXMudmFsdWVzKCldCiAgICAgICAgc2VsZi5zdGVwID0gX2djZF9saXN0KG1hZ3MpIGlmIG1hZ3MgZWxzZSAxCiAgICAgICAgc2VsZi5vcmlnaW46IHR1cGxlW2Zsb2F0LCBmbG9hdF0gfCBOb25lID0gTm9uZQogICAgICAgIHNlbGYuZnJlZTogc2V0W3R1cGxlW2ludCwgaW50XV0gPSBzZXQoKQogICAgICAgIHNlbGYuYmxvY2tlZDogc2V0W3R1cGxlW2ludCwgaW50XV0gPSBzZXQoKQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIHVzYWJsZShzZWxmKSAtPiBib29sOgogICAgICAgICMgbmVlZCBheGlzLWFsaWduZWQgbW92ZXMgY292ZXJpbmcgYm90aCBheGVzIHRvIHBhdGhmaW5kIGluIDJECiAgICAgICAgaGF2ZXNfcm93ID0gYW55KGRyICE9IDAgZm9yIGRyLCBkYyBpbiBzZWxmLmRlbHRhcy52YWx1ZXMoKSkKICAgICAgICBoYXZlc19jb2wgPSBhbnkoZGMgIT0gMCBmb3IgZHIsIGRjIGluIHNlbGYuZGVsdGFzLnZhbHVlcygpKQogICAgICAgIHJldHVybiBzZWxmLnN0ZXAgPiAwIGFuZCBoYXZlc19yb3cgYW5kIGhhdmVzX2NvbAoKICAgIGRlZiBxdWFudGl6ZShzZWxmLCBjZW50cm9pZDogdHVwbGVbZmxvYXQsIGZsb2F0XSkgLT4gdHVwbGVbaW50LCBpbnRdOgogICAgICAgIGlmIHNlbGYub3JpZ2luIGlzIE5vbmU6CiAgICAgICAgICAgIHNlbGYub3JpZ2luID0gY2VudHJvaWQKICAgICAgICByMCwgYzAgPSBzZWxmLm9yaWdpbgogICAgICAgIHJldHVybiAocm91bmQoKGNlbnRyb2lkWzBdIC0gcjApIC8gc2VsZi5zdGVwKSwgcm91bmQoKGNlbnRyb2lkWzFdIC0gYzApIC8gc2VsZi5zdGVwKSkKCiAgICBkZWYgX3N0ZXBfdW5pdHMoc2VsZiwgZHI6IGludCwgZGM6IGludCkgLT4gdHVwbGVbaW50LCBpbnRdOgogICAgICAgIHJldHVybiAoaW50KHJvdW5kKGRyIC8gc2VsZi5zdGVwKSksIGludChyb3VuZChkYyAvIHNlbGYuc3RlcCkpKQoKICAgIGRlZiBvYnNlcnZlX21vdmUoc2VsZiwgYmVmb3JlOiB0dXBsZVtmbG9hdCwgZmxvYXRdLCBhY3Rpb25faWQ6IGludCwKICAgICAgICAgICAgICAgICAgICAgYWZ0ZXI6IHR1cGxlW2Zsb2F0LCBmbG9hdF0pIC0+IE5vbmU6CiAgICAgICAgIiIiUmVjb3JkIHRoZSBvdXRjb21lIG9mIGEgc2ltcGxlIG1vdmUgZm9yIG9jY3VwYW5jeSBsZWFybmluZy4iIiIKICAgICAgICBpZiBhY3Rpb25faWQgbm90IGluIHNlbGYuZGVsdGFzOgogICAgICAgICAgICByZXR1cm4KICAgICAgICBxYiA9IHNlbGYucXVhbnRpemUoYmVmb3JlKQogICAgICAgIHFhID0gc2VsZi5xdWFudGl6ZShhZnRlcikKICAgICAgICBzZWxmLmZyZWUuYWRkKHFiKQogICAgICAgIGRyLCBkYyA9IHNlbGYuZGVsdGFzW2FjdGlvbl9pZF0KICAgICAgICB1ciwgdWMgPSBzZWxmLl9zdGVwX3VuaXRzKGRyLCBkYykKICAgICAgICB0YXJnZXQgPSAocWJbMF0gKyB1ciwgcWJbMV0gKyB1YykKICAgICAgICBpZiBxYSA9PSBxYjoKICAgICAgICAgICAgIyBhdmF0YXIgZGlkbid0IG1vdmUgLT4gdGhlIHRhcmdldCBjZWxsIGlzIGJsb2NrZWQgKHdhbGwvYm91bmRhcnkpCiAgICAgICAgICAgIHNlbGYuYmxvY2tlZC5hZGQodGFyZ2V0KQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNlbGYuZnJlZS5hZGQocWEpCgogICAgZGVmIGFzdGFyKHNlbGYsIHN0YXJ0OiB0dXBsZVtmbG9hdCwgZmxvYXRdLCBnb2FsOiB0dXBsZVtmbG9hdCwgZmxvYXRdKSAtPiBsaXN0W2ludF0gfCBOb25lOgogICAgICAgICIiIlJldHVybiBhIGxpc3Qgb2YgYWN0aW9uX2lkcyBtb3ZpbmcgdGhlIGF2YXRhciBmcm9tIHN0YXJ0IHRvIGdvYWwsIG9yIE5vbmUuCgogICAgICAgIFBsYW5zIG92ZXIgdGhlIGxhdHRpY2U6IGtub3duLWJsb2NrZWQgY2VsbHMgYXJlIHdhbGxzOyB1bmtub3duIGNlbGxzIGFyZSBhc3N1bWVkCiAgICAgICAgZnJlZSAob3B0aW1pc3RpYykuIEdvYWwgaXMgbWF0Y2hlZCBhdCBsYXR0aWNlIHJlc29sdXRpb24uCiAgICAgICAgIiIiCiAgICAgICAgaWYgbm90IHNlbGYudXNhYmxlOgogICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgIHFzID0gc2VsZi5xdWFudGl6ZShzdGFydCkKICAgICAgICBxZyA9IHNlbGYucXVhbnRpemUoZ29hbCkKICAgICAgICBpZiBxcyA9PSBxZzoKICAgICAgICAgICAgcmV0dXJuIFtdCiAgICAgICAgbW92ZXMgPSBbKGEsIHNlbGYuX3N0ZXBfdW5pdHMoZHIsIGRjKSkgZm9yIGEsIChkciwgZGMpIGluIHNlbGYuZGVsdGFzLml0ZW1zKCldCgogICAgICAgIGRlZiBoKHApOgogICAgICAgICAgICByZXR1cm4gYWJzKHBbMF0gLSBxZ1swXSkgKyBhYnMocFsxXSAtIHFnWzFdKQoKICAgICAgICBvcGVuaCA9IFsoaChxcyksIDAsIHFzLCBbXSldCiAgICAgICAgc2VlbiA9IHtxczogMH0KICAgICAgICBib3VuZCA9IDQgKiAoYWJzKHFzWzBdIC0gcWdbMF0pICsgYWJzKHFzWzFdIC0gcWdbMV0pICsgNCkgICMgYXZvaWQgcnVuYXdheSBpbiBvcGVuIHNwYWNlCiAgICAgICAgd2hpbGUgb3Blbmg6CiAgICAgICAgICAgIGYsIGcsIHBvcywgcGF0aCA9IGhlYXBxLmhlYXBwb3Aob3BlbmgpCiAgICAgICAgICAgIGlmIHBvcyA9PSBxZzoKICAgICAgICAgICAgICAgIHJldHVybiBwYXRoCiAgICAgICAgICAgIGlmIGcgPiBib3VuZDoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGZvciBhLCAodXIsIHVjKSBpbiBtb3ZlczoKICAgICAgICAgICAgICAgIG5wb3MgPSAocG9zWzBdICsgdXIsIHBvc1sxXSArIHVjKQogICAgICAgICAgICAgICAgaWYgbnBvcyBpbiBzZWxmLmJsb2NrZWQ6CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIG5nID0gZyArIDEKICAgICAgICAgICAgICAgIGlmIG5wb3MgaW4gc2VlbiBhbmQgc2VlbltucG9zXSA8PSBuZzoKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgc2VlbltucG9zXSA9IG5nCiAgICAgICAgICAgICAgICBoZWFwcS5oZWFwcHVzaChvcGVuaCwgKG5nICsgaChucG9zKSwgbmcsIG5wb3MsIHBhdGggKyBbYV0pKQogICAgICAgIHJldHVybiBOb25lCgoKZGVmIF9nY2RfbGlzdCh4czogbGlzdFtpbnRdKSAtPiBpbnQ6CiAgICBnID0gMAogICAgZm9yIHggaW4geHM6CiAgICAgICAgZyA9IGdjZChnLCBpbnQoeCkpCiAgICByZXR1cm4gZyBvciAxCg=='}
for _name, _b in PKG.items():
    pathlib.Path('/kaggle/working/arcagi3', _name).write_bytes(base64.b64decode(_b))
print('wrote arcagi3 package:', sorted(PKG))


In [ ]:
%%writefile /kaggle/working/my_agent.py
# =====================================================================
# ARC-AGI-3 submission agent (ARC Prize 2026) — fail-safe adapter.
#
# Wraps arcagi3.policy.HybridPolicy in the official Agent interface. The arcagi3 package is
# written to /kaggle/working/arcagi3 by the notebook (self-contained) and/or attached as the
# arcagi3-agent dataset; we add every plausible path to sys.path. If the package cannot be
# imported for ANY reason, we fall back to a built-in random policy so the agent ALWAYS acts
# (a scored submission beats an ERROR, and tells us import was the issue).
# =====================================================================
import os
import random
import sys
import time
import traceback

_CANDIDATES = [
    "/kaggle/working",                  # notebook writes arcagi3/ here (primary, self-contained)
    "/kaggle/working/ARC-AGI-3-Agents/agents/templates",
    "/kaggle/input/arcagi3-agent",      # dataset fallback
    "/kaggle/input/arcagi3-agent/src",
    "/kaggle/input/arcagi3",
    os.path.dirname(os.path.abspath(__file__)),
]
for _p in _CANDIDATES:
    if _p and os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)

_HybridPolicy = None
_IMPORT_ERR = None
try:
    from arcagi3.policy import HybridPolicy as _HybridPolicy
except Exception as _e:  # noqa: BLE001
    _IMPORT_ERR = "".join(traceback.format_exception(type(_e), _e, _e.__traceback__))
    print(f"[my_agent] arcagi3 import FAILED -> random fallback.\n{_IMPORT_ERR}", flush=True)

try:
    from agents.agent import Agent as _BaseAgent
except Exception:  # local dev / stub
    class _BaseAgent:  # minimal stub
        def __init__(self, *a, **k):
            self.game_id = k.get("game_id", "?")
            self.action_counter = 0
            self.frames = []

from arcengine import GameAction as _GA  # noqa: E402
from arcengine import GameState as _GS  # noqa: E402

_TIME_BUDGET_S = 8 * 3600 - 5 * 60


class MyAgent(_BaseAgent):
    """Hybrid explorer (fail-safe). Falls back to random actions if arcagi3 is unavailable."""

    MAX_ACTIONS = float("inf")

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._pol = _HybridPolicy() if _HybridPolicy is not None else None
        self._t0 = time.time()
        seed = int(time.time() * 1e6) % (2 ** 32 - 1)
        random.seed(seed)

    def is_done(self, frames, latest_frame):
        try:
            if latest_frame.state is _GS.WIN:
                return True
        except Exception:
            pass
        return (time.time() - self._t0) >= _TIME_BUDGET_S

    def _avail_ids(self, latest_frame):
        out = []
        for a in (getattr(latest_frame, "available_actions", None) or []):
            try:
                out.append(a.value if hasattr(a, "value") else int(a))
            except Exception:
                pass
        return out

    def _random_action(self, latest_frame):
        st = getattr(latest_frame, "state", None)
        if st is _GS.NOT_PLAYED or st is _GS.GAME_OVER:
            a = _GA.RESET
            a.reasoning = "reset"
            return a
        avail = self._avail_ids(latest_frame) or [1, 2, 3, 4]
        aid = random.choice(avail)
        if aid == 6:
            a = _GA.ACTION6
            a.set_data({"x": random.randint(0, 63), "y": random.randint(0, 63)})
            a.reasoning = "random click"
            return a
        a = _GA.from_id(aid)
        a.reasoning = "random"
        return a

    def choose_action(self, frames, latest_frame):
        if self._pol is None:
            return self._random_action(latest_frame)
        try:
            import numpy as np

            arr = np.asarray(latest_frame.frame, dtype=np.int8)
            grid = arr[-1] if arr.ndim == 3 else arr
            st = latest_frame.state
            token = self._pol.decide(
                grid,
                gstate_terminal=(st is _GS.GAME_OVER),
                gstate_notplayed=(st is _GS.NOT_PLAYED),
                levels=int(getattr(latest_frame, "levels_completed", 0) or 0),
                available=self._avail_ids(latest_frame),
            )
            if token[0] == "reset":
                a = _GA.RESET
                a.reasoning = "reset"
                return a
            if token[0] == "S":
                a = _GA.from_id(token[1])
                a.reasoning = "hybrid"
                return a
            a = _GA.ACTION6
            a.set_data({"x": int(token[1]), "y": int(token[2])})
            a.reasoning = "hybrid-click"
            return a
        except Exception as e:  # never die mid-game
            print(f"[my_agent] choose_action error -> random: {e}", flush=True)
            return self._random_action(latest_frame)


In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # 1) wait for the gateway that serves the private games
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # 2) copy the official agents framework to a writable location
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    # 3) drop our agent into the framework templates (it imports arcagi3 from /kaggle/working)
    !cp /kaggle/working/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    # 4) minimal agents/__init__.py: register only what we need (avoid heavy template imports)
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write('''from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "myagent": MyAgent,
}
''')

    # 5) .env pointing the framework at the gateway (online mode, no local env files)
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write('''SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
''')

    # 6) play all games; the gateway records the scorecard -> submission
    !cd /kaggle/working/ARC-AGI-3-Agents && MPLBACKEND=agg python main.py --agent myagent


In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import pandas as pd
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 0]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    submission.head()
